## Architectural principles

The FE pipeline should follow these rules:

1. **One canonical raster space**
   - source-aligned features
   - modeling-ready raster stack
   - patch export / chunk manifests

2. **One canonical object space**
   - shrub objects from labeling
   - aggregated feature summaries
   - provenance / confidence / QA fields

3. **Chunk is the primary compute unit**
   - not whole-site full-raster computation by default

4. **Manifest-first resume**
   - stages and families write manifests
   - large artifacts may be pushed remotely and pruned locally

5. **Source-agnostic processing**
   - each source follows the same ingest → align → family → finalize contract

6. **Notebook functions should migrate cleanly to pipeline.py**

# Setup

In [83]:
# Must run on new server launch
#!pip install -r requirements.txt

In [84]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not ((repo_root / "Final").exists() and (repo_root / "Sprint 3").exists()):
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate repo root.")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


from __future__ import annotations

import json
import math
import os
import logging
import shutil
import tempfile
import time
import warnings
from collections import defaultdict
from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.windows import Window
from rasterio.warp import reproject

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from Final.config import default_config
from Final.models import (
    PipelineDomain,
    RepresentationTarget,
    SpatialScope,
    ResolutionScope,
    AvailabilityTier,
    RuntimeTier,
    ModuleCard,
    PipelineSpec,
    StageSpec,
    ModuleSpec,
    SearchAxis,
    CachePolicy,
    CacheRetentionMode,
    ArtifactSpec,
    StorageTier,
    CanonicalRasterOutputs,
    CanonicalObjectOutputs,
)
from Final.pipeline_caching import (
    hash_payload,
    write_stage_cache_manifest,
    read_stage_cache_manifest,
    is_valid_stage_cache,
)
from Final.artifact_store import (
    LocalArtifactStore,
    DriveRegistryArtifactStore,
    HybridArtifactStore,
)

from Final.labeling.io import download_file, extract_als_metadata
from Final.labeling.manifests import list_files_with_suffix, site_to_remote_base, site_to_tif_name

from Final.features.fe_2d import (
    get_color_diff,
    get_entropy_feature,
    get_fast_lbp_texture,
    get_fractal_dimension_map,
    get_granulometry_features,
    get_ldp_feature,
    get_uniform_blur,
    get_wavelet_features,
    fast_cpu_gabor,
)
from scipy.ndimage import grey_opening, grey_closing

from Final.features.fe_3dep import (
    get_slope_and_aspect,
    get_northness_eastness,
    get_curvature,
    get_ruggedness,
    get_tpi,
    get_exposure_proxies,
)

from Final.features.fe_rap import (
    get_surrounding_shrub_fraction,
    get_broad_vegetation_composition,
    extract_coarse_vegetation_prior,
)

cfg = default_config()
cfg.ensure_dirs()


LOGGER = logging.getLogger("features.notebook")
if not LOGGER.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    )

cfg.output.features_root.mkdir(parents=True, exist_ok=True)
cfg.output.logs_root.mkdir(parents=True, exist_ok=True)

print("Sites:", cfg.sites)
print("Project root:", cfg.data.project_root)
print("Features root:", cfg.output.features_root)
print("Canonical grid source:", cfg.features.canonical_grid_source)
print("Object table enabled:", cfg.features.object_table_enabled)

Sites: ['calaveras-big-trees', 'dl-bliss', 'independence-lake', 'pacific-union-college', 'sedgwick', 'shaver-lake']
Project root: /home/jovyan/work/Dry-shRub/shrub
Features root: /home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features
Canonical grid source: naip
Object table enabled: True


# Run Context

In [85]:
NOTEBOOK_VERSION = "fe_notebook"

DEFAULT_SOURCE_ORDER = ("naip", "als", "3dep", "rap")

DEFAULT_FAMILY_ORDER = {
    "naip": (
        "raw",
        "veg_idx",
        "texture",
        "multiscale",
        "color_diff",
        "granulometry",
        "fractal",
        "wavelet",
        "gabor",
        "ldp",
        "morphology",
    ),
    "als": ("height", "structure", "canopy_context"),
    "3dep": ("terrain", "terrain_context"),
    "rap": ("prior",),
}

@dataclass
class TimingConfig:
    enabled: bool = True
    log_start_end: bool = True
    warn_if_seconds_over: float = 30.0
    enable_rate_tracking: bool = True
    extrapolation_unit: str = "megapixels"


@dataclass
class ChunkingConfig:
    enabled: bool = True
    chunk_size_px: int = 1024
    halo_px_default: int = 32
    allow_full_raster_fallback: bool = True
    materialize_full_rasters_by_default: bool = False


@dataclass
class StorageConfig:
    enable_local_store: bool = True
    enable_drive_store: bool = False
    use_hybrid_store: bool = False

    local_storage_root: Path = cfg.output.features_root / "artifact_store_local"
    drive_registry_path: Path = cfg.data.project_root / "Final" / "artifact_registry.yaml"
    drive_config_path: Path = cfg.data.project_root / "drive_config.yaml"
    client_secrets_path: Path = cfg.data.project_root / "client_secrets.json"
    credentials_path: Path = cfg.data.project_root / "pydrive_credentials.json"

    fail_if_drive_missing: bool = False


@dataclass
class PersistenceConfig:
    persist_manifests: bool = True
    persist_chunk_outputs: bool = True
    persist_family_outputs: bool = True
    persist_stack_registry: bool = True
    persist_object_tables: bool = True
    write_runtime_csv: bool = True
    write_summary_csvs: bool = True

    push_large_artifacts_to_remote: bool = False
    prune_local_after_remote_push: bool = False
    verify_remote_before_prune: bool = True


@dataclass
class ObjectAggregationConfig:
    enabled: bool = True
    include_centroid_sample: bool = True
    centroid_only: bool = False
    square_window_radius_px: int = 3
    use_radius_scaled_window: bool = True
    min_radius_px: int = 2
    max_radius_px: int = 12
    stats: tuple[str, ...] = ("mean", "std", "min", "max")


@dataclass
class NAIPFamilyConfig:
    enabled_families: tuple[str, ...] = ("raw", "veg_idx", "texture", "multiscale")
    heavy_families_enabled: tuple[str, ...] = ()

    texture_entropy_window: int = 9
    texture_lbp_radius: int = 2
    texture_blur_sizes: tuple[int, ...] = (5, 11)

    multiscale_sizes: tuple[int, ...] = (3, 7, 15)

    granulometry_scales: tuple[int, ...] = (3, 5, 7, 11)
    fractal_scales: tuple[int, ...] = (3, 5, 7, 9)
    wavelet_type: str = "db2"
    wavelet_level: int = 2
    gabor_frequencies: tuple[float, ...] = (0.1, 0.2)
    gabor_thetas: tuple[float, ...] = (0.0, math.pi / 4, math.pi / 2, 3 * math.pi / 4)
    ldp_k: int = 3
    morphology_window_sizes: tuple[int, ...] = (3, 7, 11)

    quantize_for_texture: bool = True
    quantize_levels: int = 256
    base_band_preference: tuple[str, ...] = ("nir", "green", "red", "blue")


@dataclass
class ALSFamilyConfig:
    enabled_families: tuple[str, ...] = ("height",)
    heavy_families_enabled: tuple[str, ...] = ()

    canopy_height_resolution_m: float = 1.0
    distance_to_tall_canopy_threshold_m: float = 5.0
    local_relief_window_px: int = 5
    knn_k: int = 30


@dataclass
class TerrainFamilyConfig:
    enabled_families: tuple[str, ...] = ("terrain",)
    heavy_families_enabled: tuple[str, ...] = ()
    ruggedness_kernel_radius: int = 3
    tpi_radius_meters: int = 150
    hillshade_azimuth: int = 270
    hillshade_zenith: int = 45


@dataclass
class RAPFamilyConfig:
    enabled_families: tuple[str, ...] = ("prior",)
    context_radius_meters: int = 500


@dataclass
class NotebookFEConfig:
    sites: list[str] = field(default_factory=lambda: list(cfg.sites))
    enabled_sources: dict[str, bool] = field(
        default_factory=lambda: {k: True for k in DEFAULT_SOURCE_ORDER}
    )
    source_order: tuple[str, ...] = DEFAULT_SOURCE_ORDER

    canonical_grid_source: str = cfg.features.canonical_grid_source
    object_table_enabled: bool = cfg.features.object_table_enabled

    force_refresh_assets: bool = False
    force_refresh_features: bool = False
    force_refresh_objects: bool = False
    force_rebuild_chunk_manifest: bool = False

    timing: TimingConfig = field(default_factory=TimingConfig)
    chunking: ChunkingConfig = field(default_factory=ChunkingConfig)
    storage: StorageConfig = field(default_factory=StorageConfig)
    persistence: PersistenceConfig = field(default_factory=PersistenceConfig)
    object_agg: ObjectAggregationConfig = field(default_factory=ObjectAggregationConfig)

    naip: NAIPFamilyConfig = field(default_factory=NAIPFamilyConfig)
    als: ALSFamilyConfig = field(default_factory=ALSFamilyConfig)
    terrain: TerrainFamilyConfig = field(default_factory=TerrainFamilyConfig)
    rap: RAPFamilyConfig = field(default_factory=RAPFamilyConfig)

    cache_root: Path = cfg.output.features_root / "notebook_cache"
    summary_root: Path = cfg.output.features_root / "notebook_summaries"
    qa_root: Path = cfg.output.features_root / "notebook_qa"

    version: str = NOTEBOOK_VERSION

    def resolve(self) -> "NotebookFEConfig":
        self.cache_root = Path(self.cache_root).resolve()
        self.summary_root = Path(self.summary_root).resolve()
        self.qa_root = Path(self.qa_root).resolve()
        self.storage.local_storage_root = Path(self.storage.local_storage_root).resolve()

        for root in [
            self.cache_root,
            self.summary_root,
            self.qa_root,
            self.storage.local_storage_root,
        ]:
            root.mkdir(parents=True, exist_ok=True)

        return self


fe_cfg = NotebookFEConfig().resolve()
LOGGER.info("Resolved notebook FE config")

2026-04-17 03:54:56,805 | INFO     | features.notebook | Resolved notebook FE config


In [86]:
RUNTIME_LOGS: list[dict[str, Any]] = []
RUNTIME_ACCUM = defaultdict(list)


def log_info(msg: str, *args):
    LOGGER.info(msg, *args)


def stable_json(payload: dict[str, Any]) -> str:
    return json.dumps(payload, sort_keys=True, default=str)


def config_signature(payload: dict[str, Any]) -> str:
    return hash_payload(payload)


def fe_config_signature() -> str:
    return config_signature(asdict(fe_cfg))


def runtime_frame() -> pd.DataFrame:
    return pd.DataFrame(RUNTIME_LOGS) if RUNTIME_LOGS else pd.DataFrame()


def array_megapixels(arr: np.ndarray) -> float:
    return float(arr.shape[0] * arr.shape[1]) / 1_000_000.0


def estimate_remaining_time(stage_name: str, remaining_units: float) -> float | None:
    rows = RUNTIME_ACCUM.get(stage_name, [])
    rates = [
        r["sec_per_unit"]
        for r in rows
        if np.isfinite(r.get("sec_per_unit", np.nan))
    ]
    if not rates:
        return None
    return float(np.mean(rates) * remaining_units)


@contextmanager
def timed_stage(
    stage_name: str,
    *,
    site_id: str | None = None,
    source_name: str | None = None,
    family_name: str | None = None,
    unit_amount: float | None = None,
    extra: dict[str, Any] | None = None,
):
    start = time.perf_counter()

    if fe_cfg.timing.log_start_end:
        log_info(
            "START | stage=%s | site=%s | source=%s | family=%s",
            stage_name, site_id, source_name, family_name
        )

    try:
        yield
    finally:
        duration = time.perf_counter() - start
        sec_per_unit = duration / unit_amount if (unit_amount is not None and unit_amount > 0) else np.nan

        row = {
            "stage_name": stage_name,
            "site_id": site_id,
            "source_name": source_name,
            "family_name": family_name,
            "duration_sec": duration,
            "unit_amount": unit_amount,
            "sec_per_unit": sec_per_unit,
            **(extra or {}),
        }
        RUNTIME_LOGS.append(row)
        RUNTIME_ACCUM[stage_name].append(row)

        suffix = ""
        if np.isfinite(sec_per_unit):
            suffix = f" | sec_per_unit={sec_per_unit:.4f}"

        log_info(
            "END   | stage=%s | site=%s | source=%s | family=%s | duration=%.2fs%s",
            stage_name, site_id, source_name, family_name, duration, suffix
        )

        if duration >= fe_cfg.timing.warn_if_seconds_over:
            log_info(
                "SLOW  | stage=%s | duration=%.2fs exceeded threshold=%.2fs",
                stage_name, duration, fe_cfg.timing.warn_if_seconds_over
            )


print("FE config signature:", fe_config_signature())

FE config signature: 02dd149b925aa1d9


## Artifact Storage

In [87]:
def validate_drive_files(storage_cfg: StorageConfig) -> pd.DataFrame:
    rows = []
    for label, path in [
        ("client_secrets.json", storage_cfg.client_secrets_path),
        ("drive_config.yaml", storage_cfg.drive_config_path),
        ("pydrive_credentials.json", storage_cfg.credentials_path),
    ]:
        rows.append(
            {
                "file": label,
                "path": str(path),
                "exists": Path(path).exists(),
            }
        )
    return pd.DataFrame(rows)


def build_local_artifact_store(storage_cfg: StorageConfig) -> LocalArtifactStore:
    return LocalArtifactStore(
        repo_root=cfg.data.project_root,
        storage_root=storage_cfg.local_storage_root,
    )


def maybe_build_drive_artifact_store(storage_cfg: StorageConfig) -> DriveRegistryArtifactStore | None:
    if not storage_cfg.enable_drive_store:
        return None

    missing = []
    for p in [storage_cfg.client_secrets_path, storage_cfg.drive_config_path]:
        if not Path(p).exists():
            missing.append(str(p))

    if missing:
        msg = f"Missing Drive config files: {missing}"
        if storage_cfg.fail_if_drive_missing:
            raise FileNotFoundError(msg)
        LOGGER.warning(msg)
        return None

    return DriveRegistryArtifactStore(
        repo_root=cfg.data.project_root,
        registry_path=storage_cfg.drive_registry_path,
        drive_config_path=storage_cfg.drive_config_path,
        client_secrets_path=storage_cfg.client_secrets_path,
        credentials_path=storage_cfg.credentials_path,
    )


def build_artifact_store(storage_cfg: StorageConfig):
    local_store = build_local_artifact_store(storage_cfg)
    drive_store = maybe_build_drive_artifact_store(storage_cfg)

    if storage_cfg.use_hybrid_store and drive_store is not None:
        LOGGER.info("Using HybridArtifactStore")
        return HybridArtifactStore(local_store=local_store, remote_store=drive_store)

    if drive_store is not None and not storage_cfg.enable_local_store:
        LOGGER.info("Using DriveRegistryArtifactStore only")
        return drive_store

    LOGGER.info("Using LocalArtifactStore only")
    return local_store


artifact_store = build_artifact_store(fe_cfg.storage)

display(validate_drive_files(fe_cfg.storage))
print("Artifact store type:", type(artifact_store).__name__)

2026-04-17 03:54:59,576 | INFO     | features.notebook | Using LocalArtifactStore only


,file,path,exists
0,client_secrets.json,/home/jovyan/work/Dry-shRub/shrub/client_secre...,True
1,drive_config.yaml,/home/jovyan/work/Dry-shRub/shrub/drive_config...,True
2,pydrive_credentials.json,/home/jovyan/work/Dry-shRub/shrub/pydrive_cred...,True


Artifact store type: LocalArtifactStore


In [88]:
def drive_root_id_from_config(path: str | Path) -> str | None:
    path = Path(path)
    if not path.exists():
        return None
    try:
        import yaml
        payload = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
        return payload.get("drive_root_folder_id")
    except Exception as e:
        LOGGER.warning("Failed reading drive config %s: %s", path, e)
        return None


def drive_status_frame(storage_cfg: StorageConfig) -> pd.DataFrame:
    rows = []

    rows.append(
        {
            "check": "client_secrets_exists",
            "value": Path(storage_cfg.client_secrets_path).exists(),
            "detail": str(storage_cfg.client_secrets_path),
        }
    )
    rows.append(
        {
            "check": "drive_config_exists",
            "value": Path(storage_cfg.drive_config_path).exists(),
            "detail": str(storage_cfg.drive_config_path),
        }
    )
    rows.append(
        {
            "check": "credentials_exists",
            "value": Path(storage_cfg.credentials_path).exists(),
            "detail": str(storage_cfg.credentials_path),
        }
    )
    rows.append(
        {
            "check": "drive_root_folder_id_present",
            "value": drive_root_id_from_config(storage_cfg.drive_config_path) is not None,
            "detail": str(drive_root_id_from_config(storage_cfg.drive_config_path)),
        }
    )

    return pd.DataFrame(rows)


display(drive_status_frame(fe_cfg.storage))

,check,value,detail
0,client_secrets_exists,True,/home/jovyan/work/Dry-shRub/shrub/client_secre...
1,drive_config_exists,True,/home/jovyan/work/Dry-shRub/shrub/drive_config...
2,credentials_exists,True,/home/jovyan/work/Dry-shRub/shrub/pydrive_cred...
3,drive_root_folder_id_present,True,1F9AtiUfx_z48tQIkNbDpzZUpH6R5uoCN


In [89]:
def smoke_test_artifact_store_roundtrip(store, *, rel_path: str = "features/_smoke_test/roundtrip.txt") -> dict[str, Any]:
    temp_dir = Path(tempfile.mkdtemp(prefix="fe_artifact_smoke_"))
    local_path = temp_dir / "roundtrip.txt"
    local_path.write_text("artifact store smoke test", encoding="utf-8")

    result = {
        "store_type": type(store).__name__,
        "rel_path": rel_path,
        "push_ok": False,
        "pull_ok": False,
        "content_match": False,
        "remote_ref": None,
        "pulled_path": None,
    }

    try:
        remote_ref = store.push(local_path, rel_path=rel_path)
        result["push_ok"] = True
        result["remote_ref"] = remote_ref

        local_path.unlink(missing_ok=True)

        pulled_path = store.pull(rel_path, local_path=local_path)
        result["pull_ok"] = True
        result["pulled_path"] = str(pulled_path)

        content = Path(pulled_path).read_text(encoding="utf-8")
        result["content_match"] = (content == "artifact store smoke test")
    except Exception as e:
        result["error"] = str(e)
    finally:
        try:
            shutil.rmtree(temp_dir, ignore_errors=True)
        except Exception:
            pass

    return result


#smoke_result = smoke_test_artifact_store_roundtrip(artifact_store)
#display(pd.DataFrame([smoke_result]))

# Contracts

In [90]:
@dataclass
class SiteAssetBundle:
    site_id: str
    source_assets: dict[str, Any] = field(default_factory=dict)
    notes: list[str] = field(default_factory=list)


@dataclass
class CanonicalGrid:
    site_id: str
    width: int
    height: int
    transform: Affine
    crs: Any
    source_name: str
    nodata: float | int | None = None

    @property
    def pixel_size(self) -> tuple[float, float]:
        return abs(self.transform.a), abs(self.transform.e)

    def profile(self, *, dtype: str = "float32", count: int = 1, nodata: float | int | None = None) -> dict[str, Any]:
        return {
            "driver": "GTiff",
            "width": self.width,
            "height": self.height,
            "count": count,
            "dtype": dtype,
            "crs": self.crs,
            "transform": self.transform,
            "nodata": self.nodata if nodata is None else nodata,
        }


@dataclass
class ChunkRecord:
    site_id: str
    chunk_id: str
    row_start: int
    row_end: int
    col_start: int
    col_end: int
    halo_px: int

    @property
    def height(self) -> int:
        return self.row_end - self.row_start

    @property
    def width(self) -> int:
        return self.col_end - self.col_start


@dataclass
class ChunkManifest:
    site_id: str
    chunk_size_px: int
    halo_px_default: int
    records: list[ChunkRecord]


@dataclass
class SourceRasterBundle:
    site_id: str
    source_name: str
    arrays: dict[str, np.ndarray]
    transform: Affine
    crs: Any
    nodata: float | int | None = None
    metadata: dict[str, Any] = field(default_factory=dict)

    @property
    def shape(self) -> tuple[int, int]:
        first = next(iter(self.arrays.values()))
        return first.shape


@dataclass
class FeatureFamilyResult:
    site_id: str
    source_name: str
    family_name: str
    arrays: dict[str, np.ndarray]
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass
class RasterLayerRecord:
    site_id: str
    source_name: str
    family_name: str
    layer_name: str
    rel_path: str | None = None
    local_path: str | None = None
    remote_ref: str | None = None
    chunked: bool = True
    dtype: str | None = None
    shape: tuple[int, int] | None = None
    notes: list[str] = field(default_factory=list)


@dataclass
class RasterStackRegistry:
    site_id: str
    config_signature: str
    layers: list[RasterLayerRecord] = field(default_factory=list)


@dataclass
class ObjectFeatureRegistry:
    site_id: str
    config_signature: str
    object_table_path: str | None = None
    notes: list[str] = field(default_factory=list)

In [91]:
def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def notebook_cache_root() -> Path:
    return ensure_dir(fe_cfg.cache_root)


def notebook_stage_cache_root(stage_name: str) -> Path:
    return ensure_dir(notebook_cache_root() / stage_name)


def stage_cache_dir(
    *,
    stage_name: str,
    site_id: str,
    source_name: str | None = None,
    data_signature: str | None = None,
    config_signature: str | None = None,
) -> Path:
    root = ensure_dir(notebook_stage_cache_root(stage_name) / site_id)
    if source_name is not None:
        root = ensure_dir(root / source_name)
    if data_signature is not None and config_signature is not None:
        root = ensure_dir(root / f"{data_signature}__{config_signature}")
    return root


def write_json(path: str | Path, payload: dict[str, Any]) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    return path


def read_json(path: str | Path) -> dict[str, Any]:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def summarize_array(name: str, arr: np.ndarray) -> dict[str, Any]:
    finite = np.isfinite(arr)
    return {
        "name": name,
        "shape": arr.shape,
        "finite_fraction": float(finite.mean()) if finite.size else np.nan,
        "min": float(np.nanmin(arr)) if finite.any() else np.nan,
        "max": float(np.nanmax(arr)) if finite.any() else np.nan,
        "mean": float(np.nanmean(arr)) if finite.any() else np.nan,
        "std": float(np.nanstd(arr)) if finite.any() else np.nan,
    }


def feature_stack_frame(layer_dict: dict[str, np.ndarray]) -> pd.DataFrame:
    return pd.DataFrame([summarize_array(k, v) for k, v in layer_dict.items()]).sort_values("name").reset_index(drop=True)

In [92]:
def build_feature_pipeline_spec() -> PipelineSpec:
    modules = {
        "features.site_assets": ModuleSpec(
            key="features.site_assets",
            stage_name="site_assets",
        ),
        "features.canonical_grid": ModuleSpec(
            key="features.canonical_grid",
            stage_name="canonical_grid",
            param_keys=("canonical_grid_source",),
        ),
        "features.chunk_manifest": ModuleSpec(
            key="features.chunk_manifest",
            stage_name="chunk_manifest",
            enabled_key=None,
            param_keys=("chunking",),
        ),
        "features.source_ingest": ModuleSpec(
            key="features.source_ingest",
            stage_name="source_ingest",
            param_keys=("enabled_sources",),
        ),
        "features.source_align": ModuleSpec(
            key="features.source_align",
            stage_name="source_align",
        ),
        "features.family_compute": ModuleSpec(
            key="features.family_compute",
            stage_name="family_compute",
        ),
        "features.stack_finalize": ModuleSpec(
            key="features.stack_finalize",
            stage_name="stack_finalize",
        ),
        "features.object_aggregate": ModuleSpec(
            key="features.object_aggregate",
            stage_name="object_aggregate",
            enabled_key="object_table_enabled",
        ),
        "features.qa_summary": ModuleSpec(
            key="features.qa_summary",
            stage_name="qa_summary",
        ),
    }

    stages = [
        StageSpec(
            name="site_assets",
            module_keys=["features.site_assets"],
            cache_policy=CachePolicy(
                require_manifest=True,
                allow_legacy_reuse=False,
                retention_mode=CacheRetentionMode.LEAN,
            ),
        ),
        StageSpec(
            name="canonical_grid",
            module_keys=["features.canonical_grid"],
            cache_policy=CachePolicy(
                require_manifest=True,
                allow_legacy_reuse=False,
                retention_mode=CacheRetentionMode.LEAN,
            ),
        ),
        StageSpec(
            name="chunk_manifest",
            module_keys=["features.chunk_manifest"],
            cache_policy=CachePolicy(
                require_manifest=True,
                allow_legacy_reuse=False,
                retention_mode=CacheRetentionMode.LEAN,
            ),
        ),
        StageSpec(
            name="source_ingest",
            module_keys=["features.source_ingest"],
            cache_policy=CachePolicy(
                require_manifest=True,
                allow_legacy_reuse=False,
                retention_mode=CacheRetentionMode.LEAN,
            ),
        ),
        StageSpec(
            name="source_align",
            module_keys=["features.source_align"],
            cache_policy=CachePolicy(
                require_manifest=True,
                allow_legacy_reuse=False,
                retention_mode=CacheRetentionMode.LEAN,
            ),
        ),
        StageSpec(
            name="family_compute",
            module_keys=["features.family_compute"],
            cache_policy=CachePolicy(
                require_manifest=True,
                allow_legacy_reuse=False,
                retention_mode=CacheRetentionMode.LEAN,
            ),
        ),
        StageSpec(
            name="stack_finalize",
            module_keys=["features.stack_finalize"],
            cache_policy=CachePolicy(
                require_manifest=True,
                allow_legacy_reuse=False,
                retention_mode=CacheRetentionMode.LEAN,
            ),
        ),
        StageSpec(
            name="object_aggregate",
            module_keys=["features.object_aggregate"],
            cache_policy=CachePolicy(
                require_manifest=True,
                allow_legacy_reuse=False,
                retention_mode=CacheRetentionMode.LEAN,
            ),
        ),
        StageSpec(
            name="qa_summary",
            module_keys=["features.qa_summary"],
            cache_policy=CachePolicy(
                require_manifest=True,
                allow_legacy_reuse=False,
                retention_mode=CacheRetentionMode.MANIFEST_ONLY,
            ),
        ),
    ]

    search_axes = [
        SearchAxis(
            key="canonical_grid_source",
            values=["naip"],
            stage_name="canonical_grid",
            module_key="features.canonical_grid",
        ),
        SearchAxis(
            key="naip_enabled_families",
            values=[("raw", "veg_idx", "texture", "multiscale")],
            stage_name="family_compute",
            module_key="features.family_compute",
        ),
        SearchAxis(
            key="chunk_size_px",
            values=[512, 1024],
            stage_name="chunk_manifest",
            module_key="features.chunk_manifest",
        ),
    ]

    return PipelineSpec(
        pipeline_name="features",
        domain=PipelineDomain.FEATURES,
        stages=stages,
        modules=modules,
        search_axes=search_axes,
    )


FE_PIPELINE_SPEC = build_feature_pipeline_spec()

print("Pipeline:", FE_PIPELINE_SPEC.pipeline_name)
print("Stages:", [s.name for s in FE_PIPELINE_SPEC.stages])
print("Modules:", list(FE_PIPELINE_SPEC.modules.keys()))

Pipeline: features
Stages: ['site_assets', 'canonical_grid', 'chunk_manifest', 'source_ingest', 'source_align', 'family_compute', 'stack_finalize', 'object_aggregate', 'qa_summary']
Modules: ['features.site_assets', 'features.canonical_grid', 'features.chunk_manifest', 'features.source_ingest', 'features.source_align', 'features.family_compute', 'features.stack_finalize', 'features.object_aggregate', 'features.qa_summary']


In [93]:
def pipeline_spec_frame(spec: PipelineSpec) -> pd.DataFrame:
    rows = []
    for stage in spec.stages:
        rows.append(
            {
                "stage_name": stage.name,
                "module_keys": ", ".join(stage.module_keys),
                "require_manifest": stage.cache_policy.require_manifest,
                "allow_legacy_reuse": stage.cache_policy.allow_legacy_reuse,
                "retention_mode": stage.cache_policy.retention_mode.value,
                "prune_after_success": stage.cache_policy.prune_after_success,
            }
        )
    return pd.DataFrame(rows)


display(pipeline_spec_frame(FE_PIPELINE_SPEC))

,stage_name,module_keys,require_manifest,allow_legacy_reuse,retention_mode,prune_after_success
0,site_assets,features.site_assets,True,False,lean,False
1,canonical_grid,features.canonical_grid,True,False,lean,False
2,chunk_manifest,features.chunk_manifest,True,False,lean,False
3,source_ingest,features.source_ingest,True,False,lean,False
4,source_align,features.source_align,True,False,lean,False
5,family_compute,features.family_compute,True,False,lean,False
6,stack_finalize,features.stack_finalize,True,False,lean,False
7,object_aggregate,features.object_aggregate,True,False,lean,False
8,qa_summary,features.qa_summary,True,False,manifest_only,False


# Chunking

In [94]:
def chunk_manifest_cache_dir(site_id: str, data_signature: str, config_signature: str) -> Path:
    return stage_cache_dir(
        stage_name="chunk_manifest",
        site_id=site_id,
        source_name="canonical_grid",
        data_signature=data_signature,
        config_signature=config_signature,
    )


def chunk_manifest_signature(site_id: str, grid: CanonicalGrid) -> tuple[str, str]:
    data_sig = hash_payload(
        {
            "site_id": site_id,
            "width": grid.width,
            "height": grid.height,
            "transform": tuple(grid.transform)[:6],
            "crs": str(grid.crs),
            "source_name": grid.source_name,
        }
    )
    config_sig = hash_payload(
        {
            "chunk_size_px": fe_cfg.chunking.chunk_size_px,
            "halo_px_default": fe_cfg.chunking.halo_px_default,
            "version": fe_cfg.version,
        }
    )
    return data_sig, config_sig

In [95]:
def build_chunk_manifest(grid: CanonicalGrid, *, force_refresh: bool = False) -> ChunkManifest:
    data_sig, config_sig = chunk_manifest_signature(grid.site_id, grid)
    cache_dir = chunk_manifest_cache_dir(grid.site_id, data_sig, config_sig)
    manifest_json = cache_dir / "chunk_manifest.json"
    stage_manifest = cache_dir / "stage_cache_manifest.json"

    if (not force_refresh) and is_valid_stage_cache(
        stage_cache_dir=cache_dir,
        expected_stage_name="chunk_manifest",
        expected_data_signature=data_sig,
        expected_config_signature=config_sig,
    ) and manifest_json.exists():
        payload = read_json(manifest_json)
        records = [ChunkRecord(**row) for row in payload["records"]]
        LOGGER.info(
            "Using cached chunk manifest | site=%s | n_chunks=%d",
            grid.site_id, len(records)
        )
        return ChunkManifest(
            site_id=payload["site_id"],
            chunk_size_px=payload["chunk_size_px"],
            halo_px_default=payload["halo_px_default"],
            records=records,
        )

    with timed_stage(
        "chunk_manifest",
        site_id=grid.site_id,
        source_name=grid.source_name,
        unit_amount=(grid.height * grid.width) / 1_000_000.0,
    ):
        chunk_size = fe_cfg.chunking.chunk_size_px
        halo = fe_cfg.chunking.halo_px_default

        records: list[ChunkRecord] = []
        chunk_idx = 0

        for row_start in range(0, grid.height, chunk_size):
            for col_start in range(0, grid.width, chunk_size):
                row_end = min(row_start + chunk_size, grid.height)
                col_end = min(col_start + chunk_size, grid.width)

                records.append(
                    ChunkRecord(
                        site_id=grid.site_id,
                        chunk_id=f"chunk_{chunk_idx:05d}",
                        row_start=row_start,
                        row_end=row_end,
                        col_start=col_start,
                        col_end=col_end,
                        halo_px=halo,
                    )
                )
                chunk_idx += 1

        chunk_manifest = ChunkManifest(
            site_id=grid.site_id,
            chunk_size_px=chunk_size,
            halo_px_default=halo,
            records=records,
        )

        write_json(
            manifest_json,
            {
                "site_id": chunk_manifest.site_id,
                "chunk_size_px": chunk_manifest.chunk_size_px,
                "halo_px_default": chunk_manifest.halo_px_default,
                "records": [asdict(r) for r in chunk_manifest.records],
            },
        )

        write_stage_cache_manifest(
            stage_cache_dir=cache_dir,
            stage_name="chunk_manifest",
            data_signature=data_sig,
            config_signature=config_sig,
            module_variants=[],
            artifact_paths={"chunk_manifest_json": str(manifest_json)},
            success=True,
            notes=[f"Built chunk manifest with {len(records)} chunks."],
        )

        LOGGER.info(
            "Built chunk manifest | site=%s | chunk_size=%d | n_chunks=%d",
            grid.site_id, chunk_size, len(records)
        )
        return chunk_manifest

In [96]:
def chunk_manifest_frame(manifest: ChunkManifest) -> pd.DataFrame:
    return pd.DataFrame([asdict(r) for r in manifest.records])


def expanded_chunk_bounds(record: ChunkRecord, *, height: int, width: int, halo_px: int | None = None) -> dict[str, int]:
    halo = record.halo_px if halo_px is None else halo_px
    return {
        "row_start": max(0, record.row_start - halo),
        "row_end": min(height, record.row_end + halo),
        "col_start": max(0, record.col_start - halo),
        "col_end": min(width, record.col_end + halo),
    }


def chunk_window(record: ChunkRecord) -> Window:
    return Window(
        col_off=record.col_start,
        row_off=record.row_start,
        width=record.width,
        height=record.height,
    )


def expanded_chunk_window(record: ChunkRecord, *, height: int, width: int, halo_px: int | None = None) -> Window:
    b = expanded_chunk_bounds(record, height=height, width=width, halo_px=halo_px)
    return Window(
        col_off=b["col_start"],
        row_off=b["row_start"],
        width=b["col_end"] - b["col_start"],
        height=b["row_end"] - b["row_start"],
    )

## Site assets and labeling bridges

In [97]:
def site_asset_cache_root(site_id: str) -> Path:
    return ensure_dir(fe_cfg.cache_root / "site_assets" / site_id)


def site_naip_cache_root(site_id: str) -> Path:
    return ensure_dir(site_asset_cache_root(site_id) / "naip")


def site_als_cache_root(site_id: str) -> Path:
    return ensure_dir(site_asset_cache_root(site_id) / "als_metadata")


def site_3dep_cache_root(site_id: str) -> Path:
    return ensure_dir(site_asset_cache_root(site_id) / "3dep")


def site_rap_cache_root(site_id: str) -> Path:
    return ensure_dir(site_asset_cache_root(site_id) / "rap")


def site_metadata_manifest_local_path(site_id: str, config_sig: str | None = None) -> Path:
    config_sig = config_sig or current_fe_config_signature()
    rel_path = render_artifact_rel_path(
        "site_metadata_manifest",
        site_id=site_id,
        config_signature=config_sig,
    )
    return local_artifact_abs_path(rel_path)


def source_inventory_local_path(site_id: str, config_sig: str | None = None) -> Path:
    config_sig = config_sig or current_fe_config_signature()
    rel_path = render_artifact_rel_path(
        "source_inventory",
        site_id=site_id,
        config_signature=config_sig,
    )
    return local_artifact_abs_path(rel_path)


def validate_cached_raster(path: str | Path) -> bool:
    path = Path(path)
    if not path.exists():
        return False
    try:
        with rasterio.open(path) as src:
            h = max(1, min(16, src.height))
            w = max(1, min(16, src.width))
            src.read([1], window=Window(0, 0, w, h))
        return True
    except Exception as e:
        LOGGER.warning("Cached raster validation failed for %s: %s", path, e)
        return False

In [98]:
def build_source_inventory(site_id: str) -> dict[str, Any]:
    site_base = site_to_remote_base(cfg, site_id)
    inventory = {
        "site_id": site_id,
        "site_base": site_base,
        "naip": {
            "expected_name": site_to_tif_name(site_id),
            "remote_url": f"{site_base}/{cfg.data.naip_3dep_dir}/{site_to_tif_name(site_id)}",
        },
        "als": {
            "remote_url": f"{site_base}/{cfg.data.als_dir}",
            "files": [],
        },
        "3dep": {
            "remote_url": None,
        },
        "rap": {
            "remote_url": None,
        },
    }

    try:
        als_files = list_files_with_suffix(
            inventory["als"]["remote_url"],
            (".las", ".laz", ".copc.laz"),
        )
        inventory["als"]["files"] = als_files
    except Exception as e:
        inventory["als"]["error"] = str(e)

    return inventory


def persist_source_inventory(site_id: str, inventory: dict[str, Any], *, config_sig: str | None = None) -> PersistedArtifactRecord:
    return persist_json_artifact(
        inventory,
        artifact_key="source_inventory",
        site_id=site_id,
        config_sig=config_sig,
    )


def try_load_source_inventory(site_id: str, *, config_sig: str | None = None) -> dict[str, Any] | None:
    return try_load_json_artifact(
        artifact_key="source_inventory",
        site_id=site_id,
        config_sig=config_sig,
    )


def prepare_naip_asset(site_id: str, *, force_refresh: bool = False, inventory: dict[str, Any] | None = None) -> Path:
    inventory = inventory or build_source_inventory(site_id)
    naip_name = inventory["naip"]["expected_name"]
    naip_url = inventory["naip"]["remote_url"]

    local_path = site_naip_cache_root(site_id) / naip_name

    if (not force_refresh) and local_path.exists() and validate_cached_raster(local_path):
        LOGGER.info("Using cached NAIP | site=%s | path=%s", site_id, local_path)
        return local_path

    if local_path.exists():
        local_path.unlink(missing_ok=True)

    LOGGER.info("Downloading NAIP | site=%s | url=%s", site_id, naip_url)
    download_file(naip_url, local_path)

    if not validate_cached_raster(local_path):
        raise RuntimeError(f"Downloaded NAIP is unreadable for site={site_id}: {local_path}")

    return local_path


def prepare_als_metadata(
    site_id: str,
    *,
    force_refresh: bool = False,
    inventory: dict[str, Any] | None = None,
    config_sig: str | None = None,
) -> list[dict[str, Any]]:
    config_sig = config_sig or current_fe_config_signature()

    # First: try local cache in site asset dir
    out_json = site_als_cache_root(site_id) / "als_metadata.json"
    if (not force_refresh) and out_json.exists():
        LOGGER.info("Using cached ALS metadata (local site cache) | site=%s | path=%s", site_id, out_json)
        return json.loads(out_json.read_text(encoding="utf-8"))

    # Second: try remote/local artifact-store-backed manifest artifact
    payload = try_load_json_artifact(
        artifact_key="als_metadata_json",
        site_id=site_id,
        config_sig=config_sig,
    )
    if (not force_refresh) and payload is not None:
        LOGGER.info("Using cached ALS metadata (artifact store) | site=%s", site_id)
        rows = payload.get("rows", [])
        out_json.write_text(json.dumps(rows, indent=2), encoding="utf-8")
        return rows

    inventory = inventory or build_source_inventory(site_id)
    als_files = inventory.get("als", {}).get("files", [])

    if not als_files:
        LOGGER.warning("No ALS files discovered | site=%s", site_id)
        return []

    LOGGER.info("Discovered %d ALS files | site=%s", len(als_files), site_id)

    tmp_dir = Path(tempfile.mkdtemp(prefix=f"als_meta_{site_id}_"))
    rows = []

    try:
        for entry in als_files:
            local_file = tmp_dir / entry["name"]
            LOGGER.info("Downloading ALS for metadata | site=%s | file=%s", site_id, entry["name"])
            download_file(entry["url"], local_file)

            meta = extract_als_metadata(local_file)
            meta["source_file"] = entry["name"]
            rows.append(meta)

            local_file.unlink(missing_ok=True)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    out_json.write_text(json.dumps(rows, indent=2), encoding="utf-8")

    persist_json_artifact(
        {"site_id": site_id, "rows": rows},
        artifact_key="als_metadata_json",
        site_id=site_id,
        config_sig=config_sig,
    )

    LOGGER.info("Wrote ALS metadata | site=%s | path=%s", site_id, out_json)
    return rows

In [99]:
def serialize_site_asset_bundle(bundle: SiteAssetBundle) -> dict[str, Any]:
    payload = {
        "site_id": bundle.site_id,
        "source_assets": {},
        "notes": list(bundle.notes),
    }
    for key, value in bundle.source_assets.items():
        if isinstance(value, Path):
            payload["source_assets"][key] = {"kind": "path", "value": str(value)}
        else:
            payload["source_assets"][key] = {"kind": "json", "value": value}
    return payload


def deserialize_site_asset_bundle(payload: dict[str, Any]) -> SiteAssetBundle:
    bundle = SiteAssetBundle(site_id=payload["site_id"])
    bundle.notes = list(payload.get("notes", []))

    for key, wrapped in payload.get("source_assets", {}).items():
        if wrapped["kind"] == "path":
            bundle.source_assets[key] = Path(wrapped["value"])
        else:
            bundle.source_assets[key] = wrapped["value"]

    return bundle


def try_load_site_metadata_manifest(site_id: str, *, config_sig: str | None = None) -> SiteAssetBundle | None:
    payload = try_load_json_artifact(
        artifact_key="site_metadata_manifest",
        site_id=site_id,
        config_sig=config_sig,
    )
    if payload is None:
        return None
    LOGGER.info("Using site metadata manifest | site=%s", site_id)
    return deserialize_site_asset_bundle(payload)


def persist_site_metadata_manifest(bundle: SiteAssetBundle, *, config_sig: str | None = None) -> PersistedArtifactRecord:
    payload = serialize_site_asset_bundle(bundle)
    return persist_json_artifact(
        payload,
        artifact_key="site_metadata_manifest",
        site_id=bundle.site_id,
        config_sig=config_sig,
    )


def prepare_site_assets(site_id: str, *, force_refresh: bool = False) -> SiteAssetBundle:
    with timed_stage("site_assets", site_id=site_id):
        config_sig = current_fe_config_signature()

        if not force_refresh:
            existing = try_load_site_metadata_manifest(site_id, config_sig=config_sig)
            if existing is not None:
                LOGGER.info(
                    "Prepared site assets from manifest-first reuse | site=%s | keys=%s",
                    site_id, list(existing.source_assets.keys())
                )
                return existing

        inventory = try_load_source_inventory(site_id, config_sig=config_sig)
        if inventory is None or force_refresh:
            inventory = build_source_inventory(site_id)
            persist_source_inventory(site_id, inventory, config_sig=config_sig)
            LOGGER.info("Built and persisted source inventory | site=%s", site_id)
        else:
            LOGGER.info("Using cached source inventory | site=%s", site_id)

        bundle = SiteAssetBundle(site_id=site_id)

        try:
            bundle.source_assets["naip"] = prepare_naip_asset(
                site_id,
                force_refresh=force_refresh,
                inventory=inventory,
            )
        except Exception as e:
            bundle.notes.append(f"NAIP unavailable: {e}")

        try:
            bundle.source_assets["als_metadata"] = prepare_als_metadata(
                site_id,
                force_refresh=force_refresh,
                inventory=inventory,
                config_sig=config_sig,
            )
        except Exception as e:
            bundle.notes.append(f"ALS metadata unavailable: {e}")

        bundle.source_assets["source_inventory"] = inventory
        bundle.source_assets["3dep"] = None
        bundle.source_assets["rap"] = None

        persist_site_metadata_manifest(bundle, config_sig=config_sig)

        LOGGER.info(
            "Prepared site assets | site=%s | keys=%s | notes=%d",
            site_id, list(bundle.source_assets.keys()), len(bundle.notes)
        )
        return bundle

## Labeling bridge

In [100]:
def labeling_summary_dir() -> Path:
    return cfg.output.labeling_root / "summaries"


def labeling_manifest_dir() -> Path:
    return cfg.output.labeling_root / "manifests"


def labeling_pipeline_runs_root() -> Path:
    return cfg.output.labeling_root / "pipeline_runs"


def latest_labeling_run_summary_paths() -> tuple[Path | None, Path | None]:
    runs_root = labeling_pipeline_runs_root()
    if not runs_root.exists():
        return None, None

    candidate_dirs = [p for p in runs_root.iterdir() if p.is_dir()]
    candidate_dirs = sorted(candidate_dirs, key=lambda p: p.stat().st_mtime, reverse=True)

    for run_dir in candidate_dirs:
        objects_csv = run_dir / "summaries" / "objects_all.csv"
        artifacts_csv = run_dir / "summaries" / "artifacts_all.csv"
        if objects_csv.exists() and artifacts_csv.exists():
            return objects_csv, artifacts_csv

    return None, None


def normalize_site_id_value(x: Any) -> str:
    return str(x).strip().lower().replace("_", "-").replace(" ", "-")


def load_labeling_objects_summary() -> pd.DataFrame:
    # First prefer the newest config-specific run summaries
    run_objects_csv, _ = latest_labeling_run_summary_paths()
    if run_objects_csv is not None and run_objects_csv.exists():
        df = pd.read_csv(run_objects_csv)
        if "site_id" in df.columns:
            df["_site_id_norm"] = df["site_id"].map(normalize_site_id_value)
        LOGGER.info("Loaded labeling objects summary from latest pipeline run | rows=%d | path=%s", len(df), run_objects_csv)
        return df

    # Fall back to global summaries
    obj_csv = labeling_summary_dir() / "objects_all.csv"
    if obj_csv.exists():
        df = pd.read_csv(obj_csv)
        if "site_id" in df.columns:
            df["_site_id_norm"] = df["site_id"].map(normalize_site_id_value)
        LOGGER.info("Loaded labeling objects summary from global summaries | rows=%d | path=%s", len(df), obj_csv)
        return df

    LOGGER.warning("No labeling object summary found in pipeline_runs or global summaries.")
    return pd.DataFrame()


def load_labeling_artifacts_summary() -> pd.DataFrame:
    # First prefer the newest config-specific run summaries
    _, run_artifacts_csv = latest_labeling_run_summary_paths()
    if run_artifacts_csv is not None and run_artifacts_csv.exists():
        df = pd.read_csv(run_artifacts_csv)
        if "site_id" in df.columns:
            df["_site_id_norm"] = df["site_id"].map(normalize_site_id_value)
        LOGGER.info("Loaded labeling artifacts summary from latest pipeline run | rows=%d | path=%s", len(df), run_artifacts_csv)
        return df

    art_csv = labeling_summary_dir() / "artifacts_all.csv"
    if art_csv.exists():
        df = pd.read_csv(art_csv)
        if "site_id" in df.columns:
            df["_site_id_norm"] = df["site_id"].map(normalize_site_id_value)
        LOGGER.info("Loaded labeling artifacts summary from global summaries | rows=%d | path=%s", len(df), art_csv)
        return df

    manifest_csv = labeling_manifest_dir() / "sprint4_artifacts.csv"
    if manifest_csv.exists():
        df = pd.read_csv(manifest_csv)
        if "site_id" in df.columns:
            df["_site_id_norm"] = df["site_id"].map(normalize_site_id_value)
        LOGGER.info("Loaded labeling artifact manifest fallback | rows=%d | path=%s", len(df), manifest_csv)
        return df

    LOGGER.warning("No labeling artifact summary/manifest found.")
    return pd.DataFrame()


LABEL_OBJECTS_DF = load_labeling_objects_summary()
LABEL_ARTIFACTS_DF = load_labeling_artifacts_summary()

print("LABEL_OBJECTS_DF:", LABEL_OBJECTS_DF.shape)
print("LABEL_ARTIFACTS_DF:", LABEL_ARTIFACTS_DF.shape)

2026-04-17 03:55:08,715 | INFO     | features.notebook | Loaded labeling objects summary from latest pipeline run | rows=202 | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/pipeline_runs/0bab4176c29e/summaries/objects_all.csv
2026-04-17 03:55:08,884 | INFO     | features.notebook | Loaded labeling artifacts summary from latest pipeline run | rows=48 | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/pipeline_runs/0bab4176c29e/summaries/artifacts_all.csv


LABEL_OBJECTS_DF: (202, 46)
LABEL_ARTIFACTS_DF: (48, 23)


In [101]:
def label_objects_for_site(site_id: str) -> pd.DataFrame:
    if LABEL_OBJECTS_DF.empty:
        LOGGER.warning("LABEL_OBJECTS_DF is empty.")
        return pd.DataFrame()

    wanted = normalize_site_id_value(site_id)

    if "_site_id_norm" in LABEL_OBJECTS_DF.columns:
        out = LABEL_OBJECTS_DF[LABEL_OBJECTS_DF["_site_id_norm"] == wanted].copy()
    else:
        out = LABEL_OBJECTS_DF[LABEL_OBJECTS_DF["site_id"].map(normalize_site_id_value) == wanted].copy()

    LOGGER.info(
        "label_objects_for_site | requested=%s | matched_rows=%d | unique_sites=%s",
        site_id,
        len(out),
        sorted(LABEL_OBJECTS_DF["site_id"].astype(str).unique().tolist())[:20] if "site_id" in LABEL_OBJECTS_DF.columns else [],
    )
    return out


def label_artifacts_for_site(site_id: str) -> pd.DataFrame:
    if LABEL_ARTIFACTS_DF.empty:
        return pd.DataFrame()

    wanted = normalize_site_id_value(site_id)

    if "_site_id_norm" in LABEL_ARTIFACTS_DF.columns:
        return LABEL_ARTIFACTS_DF[LABEL_ARTIFACTS_DF["_site_id_norm"] == wanted].copy()

    return LABEL_ARTIFACTS_DF[LABEL_ARTIFACTS_DF["site_id"].map(normalize_site_id_value) == wanted].copy()


def best_label_artifact_for_site(site_id: str, resolution_m: float = 1.0) -> pd.Series | None:
    site_df = label_artifacts_for_site(site_id)
    if site_df.empty:
        return None

    if "resolution_m" in site_df.columns:
        site_df = site_df[np.isclose(site_df["resolution_m"].astype(float), float(resolution_m))].copy()

    if site_df.empty:
        return None

    sort_cols = [c for c in ["plot_id", "source_version"] if c in site_df.columns]
    if sort_cols:
        site_df = site_df.sort_values(sort_cols)

    return site_df.iloc[0]

In [102]:
TEST_SITE = fe_cfg.sites[0]

print("LABEL_OBJECTS_DF shape:", LABEL_OBJECTS_DF.shape)
if not LABEL_OBJECTS_DF.empty:
    print("Unique site_ids in labeling objects:")
    display(pd.Series(sorted(LABEL_OBJECTS_DF["site_id"].astype(str).unique())).to_frame("site_id"))

site_objects = label_objects_for_site(TEST_SITE)
print("site_objects shape:", site_objects.shape)
display(site_objects.head(5))

LABEL_OBJECTS_DF shape: (202, 46)
Unique site_ids in labeling objects:


,site_id
0,calaveras-big-trees
1,dl-bliss
2,independence-lake
3,pacific-union-college
4,sedgwick
5,shaver-lake


2026-04-17 03:55:10,014 | INFO     | features.notebook | label_objects_for_site | requested=calaveras-big-trees | matched_rows=36 | unique_sites=['calaveras-big-trees', 'dl-bliss', 'independence-lake', 'pacific-union-college', 'sedgwick', 'shaver-lake']


site_objects shape: (36, 46)


,site_id,plot_id,ptx_stem,ptx_date_token,ptx_date,object_id,x_tls,y_tls,height_tls,area_tls,radius_m,radius_source,n_points,valid_object,object_confidence,temporal_confidence,transform_confidence,boundary_confidence_mode,source_file,source_version,variant,label_variant,run_dir,metrics_csv,tree_inventory_csv,fuels_raster,dtm_raster,chm_raster,input_ptx,Unnamed: 0,perimeter_tls,compactness,elongation,bbox_minx,bbox_miny,bbox_maxx,bbox_maxy,dedup_keep,dedup_reason,x_als,y_als,x_naip,y_naip,row,col,_site_id_norm
0,calaveras-big-trees,CATCU_0009_20250615_1,CATCU_0009_20250615_1,20250615,2025-06-15,1,3.046618,11.705575,2.984,4.356,1.177522,observed,2170,True,1.0,NaN,1.0,uniform,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,sprint3_original,original,base,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,1,7.398588,1.0,NaN,3.046618,11.705575,3.046618,11.705575,True,NaN,743344.416325,4.237894e+06,743344.501108,4.237893e+06,48.390177,33.501847,calaveras-big-trees
1,calaveras-big-trees,CATCU_0009_20250615_1,CATCU_0009_20250615_1,20250615,2025-06-15,2,5.759161,11.264599,1.707,2.119,0.821279,observed,854,True,1.0,NaN,1.0,uniform,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,sprint3_original,original,base,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,2,5.160246,1.0,NaN,5.759161,11.264599,5.759161,11.264599,True,NaN,743344.449078,4.237897e+06,743344.533888,4.237896e+06,43.810409,33.556480,calaveras-big-trees
2,calaveras-big-trees,CATCU_0009_20250615_1,CATCU_0009_20250615_1,20250615,2025-06-15,3,9.113982,9.333501,2.987,0.199,0.251682,observed,46,True,0.8,NaN,1.0,uniform,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,sprint3_original,original,base,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,3,1.581363,1.0,NaN,9.113982,9.333501,9.113982,9.333501,True,NaN,743345.859812,4.237901e+06,743345.944654,4.237900e+06,37.803074,35.907757,calaveras-big-trees
3,calaveras-big-trees,CATCU_0009_20250615_1,CATCU_0009_20250615_1,20250615,2025-06-15,4,-8.062640,7.617650,2.994,1.773,0.751241,observed,455,True,1.0,NaN,1.0,uniform,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,sprint3_original,original,base,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,4,4.720188,1.0,NaN,-8.062640,7.617650,-8.062640,7.617650,True,NaN,743350.110286,4.237884e+06,743350.194959,4.237883e+06,65.687059,42.991599,calaveras-big-trees
4,calaveras-big-trees,CATCU_0009_20250615_1,CATCU_0009_20250615_1,20250615,2025-06-15,5,-1.666380,7.426122,2.997,0.177,0.237362,observed,74,True,0.8,NaN,1.0,uniform,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,sprint3_original,original,base,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,/home/jovyan/wo

# Canonical Grid Setup

In [103]:
def infer_naip_band_names(path: str | Path) -> list[str]:
    path = Path(path)
    with rasterio.open(path) as src:
        count = src.count

    if count == 4:
        return ["red", "green", "blue", "nir"]
    if count == 3:
        return ["red", "green", "blue"]
    return [f"band_{i}" for i in range(1, count + 1)]


def read_raster_bundle(
    path: str | Path,
    *,
    site_id: str,
    source_name: str,
    band_names: list[str] | None = None,
) -> SourceRasterBundle:
    path = Path(path)

    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
        nodata = src.nodata

        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)

        if band_names is None:
            band_names = [f"{source_name}_band_{i}" for i in range(1, src.count + 1)]

        if len(band_names) != src.count:
            raise ValueError(f"Band name count mismatch for {path}: expected {src.count}, got {len(band_names)}")

        arrays = {band_names[i]: arr[i] for i in range(src.count)}

        return SourceRasterBundle(
            site_id=site_id,
            source_name=source_name,
            arrays=arrays,
            transform=src.transform,
            crs=src.crs,
            nodata=nodata,
            metadata={
                "path": str(path),
                "width": src.width,
                "height": src.height,
                "count": src.count,
            },
        )


def canonical_grid_cache_dir(site_id: str, data_signature: str, config_signature: str) -> Path:
    return stage_cache_dir(
        stage_name="canonical_grid",
        site_id=site_id,
        source_name="naip",
        data_signature=data_signature,
        config_signature=config_signature,
    )

In [104]:
def build_canonical_grid_for_site(site_id: str, assets: SiteAssetBundle | None = None, *, force_refresh: bool = False) -> CanonicalGrid:
    if assets is None:
        assets = prepare_site_assets(site_id, force_refresh=force_refresh)

    naip_path = assets.source_assets.get("naip")
    if naip_path is None:
        raise ValueError(f"No NAIP asset available for site={site_id}; cannot build canonical grid.")

    data_sig = hash_payload(
        {
            "site_id": site_id,
            "naip_path": str(naip_path),
            "canonical_grid_source": fe_cfg.canonical_grid_source,
        }
    )
    config_sig = hash_payload(
        {
            "canonical_grid_source": fe_cfg.canonical_grid_source,
            "version": fe_cfg.version,
        }
    )
    cache_dir = canonical_grid_cache_dir(site_id, data_sig, config_sig)
    grid_json = cache_dir / "canonical_grid.json"

    if (not force_refresh) and is_valid_stage_cache(
        stage_cache_dir=cache_dir,
        expected_stage_name="canonical_grid",
        expected_data_signature=data_sig,
        expected_config_signature=config_sig,
    ) and grid_json.exists():
        payload = read_json(grid_json)
        LOGGER.info("Using cached canonical grid | site=%s", site_id)
        return CanonicalGrid(
            site_id=payload["site_id"],
            width=payload["width"],
            height=payload["height"],
            transform=Affine(*payload["transform"]),
            crs=payload["crs"],
            source_name=payload["source_name"],
            nodata=payload["nodata"],
        )

    with timed_stage("canonical_grid", site_id=site_id, source_name="naip"):
        bundle = read_raster_bundle(
            naip_path,
            site_id=site_id,
            source_name="naip",
            band_names=infer_naip_band_names(naip_path),
        )

        h, w = next(iter(bundle.arrays.values())).shape
        grid = CanonicalGrid(
            site_id=site_id,
            width=w,
            height=h,
            transform=bundle.transform,
            crs=bundle.crs,
            source_name="naip",
            nodata=np.nan,
        )

        write_json(
            grid_json,
            {
                "site_id": grid.site_id,
                "width": grid.width,
                "height": grid.height,
                "transform": list(grid.transform)[:6],
                "crs": grid.crs,
                "source_name": grid.source_name,
                "nodata": grid.nodata,
            },
        )

        write_stage_cache_manifest(
            stage_cache_dir=cache_dir,
            stage_name="canonical_grid",
            data_signature=data_sig,
            config_signature=config_sig,
            module_variants=[],
            artifact_paths={"canonical_grid_json": str(grid_json)},
            success=True,
            notes=["Built canonical grid from NAIP asset."],
        )

        LOGGER.info(
            "Built canonical grid | site=%s | shape=(%d, %d) | pixel_size=%s",
            site_id, grid.height, grid.width, grid.pixel_size
        )
        return grid

## Specs

In [105]:
@dataclass(frozen=True)
class SourceSpec:
    name: str
    native_kind: str
    default_resampling: Resampling
    notes: str = ""


SOURCE_SPECS = {
    "naip": SourceSpec(
        name="naip",
        native_kind="multiband_raster",
        default_resampling=Resampling.nearest,
        notes="Canonical appearance source and default canonical grid anchor.",
    ),
    "als": SourceSpec(
        name="als",
        native_kind="point_cloud_or_rasterized_structure",
        default_resampling=Resampling.bilinear,
        notes="Structural source for height, canopy context, and roughness.",
    ),
    "3dep": SourceSpec(
        name="3dep",
        native_kind="terrain_raster",
        default_resampling=Resampling.bilinear,
        notes="Terrain/site context source.",
    ),
    "rap": SourceSpec(
        name="rap",
        native_kind="coarse_vegetation_raster",
        default_resampling=Resampling.bilinear,
        notes="Coarse ecological prior source.",
    ),
}


def source_spec_frame() -> pd.DataFrame:
    rows = []
    for key, spec in SOURCE_SPECS.items():
        rows.append(
            {
                "source": key,
                "native_kind": spec.native_kind,
                "default_resampling": spec.default_resampling.name,
                "notes": spec.notes,
            }
        )
    return pd.DataFrame(rows)


display(source_spec_frame())

,source,native_kind,default_resampling,notes
0,naip,multiband_raster,nearest,Canonical appearance source and default canoni...
1,als,point_cloud_or_rasterized_structure,bilinear,"Structural source for height, canopy context, ..."
2,3dep,terrain_raster,bilinear,Terrain/site context source.
3,rap,coarse_vegetation_raster,bilinear,Coarse ecological prior source.


In [106]:
FE_ARTIFACT_SPECS = {
    "site_metadata_manifest": ArtifactSpec(
        key="site_metadata_manifest",
        rel_path_template="features/{site_id}/{config_signature}/site_assets/site_metadata_manifest.json",
        storage_tier=StorageTier.LOCAL_THEN_REMOTE,
        required_for_resume=True,
        prune_local_after_push=False,
    ),
    "source_inventory": ArtifactSpec(
        key="source_inventory",
        rel_path_template="features/{site_id}/{config_signature}/site_assets/source_inventory.json",
        storage_tier=StorageTier.LOCAL_THEN_REMOTE,
        required_for_resume=True,
        prune_local_after_push=False,
    ),
    "als_metadata_json": ArtifactSpec(
        key="als_metadata_json",
        rel_path_template="features/{site_id}/{config_signature}/site_assets/als_metadata.json",
        storage_tier=StorageTier.LOCAL_THEN_REMOTE,
        required_for_resume=True,
        prune_local_after_push=False,
    ),
    "chunk_manifest": ArtifactSpec(
        key="chunk_manifest",
        rel_path_template="features/{site_id}/{config_signature}/chunk_manifest/chunk_manifest.json",
        storage_tier=StorageTier.LOCAL_ONLY,
        required_for_resume=True,
        prune_local_after_push=False,
    ),
    "canonical_grid": ArtifactSpec(
        key="canonical_grid",
        rel_path_template="features/{site_id}/{config_signature}/canonical_grid/canonical_grid.json",
        storage_tier=StorageTier.LOCAL_ONLY,
        required_for_resume=True,
        prune_local_after_push=False,
    ),
    "family_chunk_npz": ArtifactSpec(
        key="family_chunk_npz",
        rel_path_template="features/{site_id}/{config_signature}/{source_name}/{family_name}/{chunk_id}.npz",
        storage_tier=StorageTier.LOCAL_THEN_REMOTE,
        required_for_resume=False,
        prune_local_after_push=True,
    ),
    "stack_registry": ArtifactSpec(
        key="stack_registry",
        rel_path_template="features/{site_id}/{config_signature}/stack/stack_registry.json",
        storage_tier=StorageTier.LOCAL_THEN_REMOTE,
        required_for_resume=True,
        prune_local_after_push=False,
    ),
    "object_feature_table": ArtifactSpec(
        key="object_feature_table",
        rel_path_template="features/{site_id}/{config_signature}/objects/object_features.csv",
        storage_tier=StorageTier.LOCAL_THEN_REMOTE,
        required_for_resume=True,
        prune_local_after_push=False,
    ),
    "runtime_log": ArtifactSpec(
        key="runtime_log",
        rel_path_template="features/{site_id}/{config_signature}/qa/runtime_log.csv",
        storage_tier=StorageTier.LOCAL_ONLY,
        required_for_resume=False,
        prune_local_after_push=False,
    ),
}

In [107]:
def artifact_spec_frame() -> pd.DataFrame:
    rows = []
    for key, spec in FE_ARTIFACT_SPECS.items():
        rows.append(
            {
                "key": key,
                "rel_path_template": spec.rel_path_template,
                "storage_tier": spec.storage_tier.value,
                "required_for_resume": spec.required_for_resume,
                "prune_local_after_push": spec.prune_local_after_push,
            }
        )
    return pd.DataFrame(rows)


display(artifact_spec_frame())

,key,rel_path_template,storage_tier,required_for_resume,prune_local_after_push
0,site_metadata_manifest,features/{site_id}/{config_signature}/site_ass...,local_then_remote,True,False
1,source_inventory,features/{site_id}/{config_signature}/site_ass...,local_then_remote,True,False
2,als_metadata_json,features/{site_id}/{config_signature}/site_ass...,local_then_remote,True,False
3,chunk_manifest,features/{site_id}/{config_signature}/chunk_ma...,local_only,True,False
4,canonical_grid,features/{site_id}/{config_signature}/canonica...,local_only,True,False
5,family_chunk_npz,features/{site_id}/{config_signature}/{source_...,local_then_remote,False,True
6,stack_registry,features/{site_id}/{config_signature}/stack/st...,local_then_remote,True,False
7,object_feature_table,features/{site_id}/{config_signature}/objects/...,local_then_remote,True,False
8,runtime_log,features/{site_id}/{config_signature}/qa/runti...,local_only,False,False


In [108]:
# Small end-to-end dry-run of the new generic runtime pieces for one site.
TEST_SITE = fe_cfg.sites[0]

site_assets = prepare_site_assets(TEST_SITE, force_refresh=fe_cfg.force_refresh_assets)
canonical_grid = build_canonical_grid_for_site(TEST_SITE, assets=site_assets, force_refresh=fe_cfg.force_refresh_features)
chunk_manifest = build_chunk_manifest(canonical_grid, force_refresh=fe_cfg.force_rebuild_chunk_manifest)

site_manifest_payload = try_load_json_artifact(
    artifact_key="site_metadata_manifest",
    site_id=TEST_SITE,
)
source_inventory_payload = try_load_json_artifact(
    artifact_key="source_inventory",
    site_id=TEST_SITE,
)

print("TEST_SITE:", TEST_SITE)
print("Asset keys:", list(site_assets.source_assets.keys()))
print("Notes:", site_assets.notes)
print("Canonical grid shape:", (canonical_grid.height, canonical_grid.width))
print("Pixel size:", canonical_grid.pixel_size)
print("Chunk count:", len(chunk_manifest.records))
print("Site manifest loaded:", site_manifest_payload is not None)
print("Source inventory loaded:", source_inventory_payload is not None)

display(chunk_manifest_frame(chunk_manifest).head(10))

2026-04-17 03:55:20,094 | INFO     | features.notebook | START | stage=site_assets | site=calaveras-big-trees | source=None | family=None
2026-04-17 03:55:20,095 | INFO     | features.notebook | LOAD ARTIFACT (local) | key=site_metadata_manifest | rel_path=features/calaveras-big-trees/02dd149b925aa1d9/site_assets/site_metadata_manifest.json
2026-04-17 03:55:20,410 | INFO     | features.notebook | Using site metadata manifest | site=calaveras-big-trees
2026-04-17 03:55:20,410 | INFO     | features.notebook | Prepared site assets from manifest-first reuse | site=calaveras-big-trees | keys=['naip', 'als_metadata', 'source_inventory', '3dep', 'rap']
2026-04-17 03:55:20,410 | INFO     | features.notebook | END   | stage=site_assets | site=calaveras-big-trees | source=None | family=None | duration=0.32s
2026-04-17 03:55:20,590 | INFO     | features.notebook | Using cached canonical grid | site=calaveras-big-trees
2026-04-17 03:55:20,735 | INFO     | features.notebook | Using cached chunk man

TEST_SITE: calaveras-big-trees
Asset keys: ['naip', 'als_metadata', 'source_inventory', '3dep', 'rap']
Notes: []
Canonical grid shape: (5375, 7398)
Pixel size: (0.6000000000000063, 0.6)
Chunk count: 48
Site manifest loaded: True
Source inventory loaded: True


,site_id,chunk_id,row_start,row_end,col_start,col_end,halo_px
0,calaveras-big-trees,chunk_00000,0,1024,0,1024,32
1,calaveras-big-trees,chunk_00001,0,1024,1024,2048,32
2,calaveras-big-trees,chunk_00002,0,1024,2048,3072,32
3,calaveras-big-trees,chunk_00003,0,1024,3072,4096,32
4,calaveras-big-trees,chunk_00004,0,1024,4096,5120,32
5,calaveras-big-trees,chunk_00005,0,1024,5120,6144,32
6,calaveras-big-trees,chunk_00006,0,1024,6144,7168,32
7,calaveras-big-trees,chunk_00007,0,1024,7168,7398,32
8,calaveras-big-trees,chunk_00008,1024,2048,0,1024,32
9,calaveras-big-trees,chunk_00009,1024,2048,1024,2048,32


## Remote Artifact Management

In [109]:
def current_fe_config_signature() -> str:
    return fe_config_signature()


def render_artifact_rel_path(
    artifact_key: str,
    *,
    site_id: str,
    config_signature: str | None = None,
    source_name: str | None = None,
    family_name: str | None = None,
    chunk_id: str | None = None,
) -> str:
    spec = FE_ARTIFACT_SPECS[artifact_key]
    config_signature = config_signature or current_fe_config_signature()
    return spec.rel_path_template.format(
        site_id=site_id,
        config_signature=config_signature,
        source_name=source_name or "unknown_source",
        family_name=family_name or "unknown_family",
        chunk_id=chunk_id or "whole",
    )


def artifact_spec_for_key(artifact_key: str) -> ArtifactSpec:
    return FE_ARTIFACT_SPECS[artifact_key]


def local_artifact_abs_path(rel_path: str) -> Path:
    if isinstance(artifact_store, LocalArtifactStore):
        return artifact_store.storage_root / rel_path
    if isinstance(artifact_store, HybridArtifactStore):
        return artifact_store.local_store.storage_root / rel_path
    return ensure_dir(fe_cfg.cache_root / "_remote_stage") / rel_path

In [110]:
@dataclass
class PersistedArtifactRecord:
    artifact_key: str
    rel_path: str
    local_path: str | None = None
    remote_ref: str | None = None
    storage_tier: str | None = None
    exists_local: bool = False
    exists_remote: bool = False
    pruned_local: bool = False
    notes: list[str] = field(default_factory=list)


def remote_artifact_exists(rel_path: str) -> bool:
    try:
        if isinstance(artifact_store, LocalArtifactStore):
            return False
        if isinstance(artifact_store, HybridArtifactStore):
            if artifact_store.remote_store is None:
                return False
            return artifact_store.remote_store.exists(rel_path)
        return artifact_store.exists(rel_path)
    except Exception as e:
        LOGGER.warning("Remote existence check failed | rel_path=%s | err=%s", rel_path, e)
        return False


def push_artifact_if_needed(
    local_path: str | Path,
    *,
    artifact_key: str,
    rel_path: str,
) -> PersistedArtifactRecord:
    spec = artifact_spec_for_key(artifact_key)
    local_path = Path(local_path)

    rec = PersistedArtifactRecord(
        artifact_key=artifact_key,
        rel_path=rel_path,
        local_path=str(local_path),
        storage_tier=spec.storage_tier.value,
        exists_local=local_path.exists(),
    )

    if not local_path.exists():
        rec.notes.append("Local artifact missing before push.")
        return rec

    should_push_remote = (
        fe_cfg.persistence.push_large_artifacts_to_remote
        and spec.storage_tier in {StorageTier.LOCAL_THEN_REMOTE, StorageTier.REMOTE_ONLY}
        and isinstance(artifact_store, (HybridArtifactStore, DriveRegistryArtifactStore))
    )

    if should_push_remote:
        LOGGER.info(
            "PUSH ARTIFACT | key=%s | rel_path=%s",
            artifact_key, rel_path
        )
        remote_ref = artifact_store.push(local_path, rel_path=rel_path)
        rec.remote_ref = remote_ref
        rec.exists_remote = True
    else:
        rec.notes.append("Remote push skipped by policy or store type.")

    return rec


def prune_local_artifact_if_allowed(
    rec: PersistedArtifactRecord,
    *,
    local_path: str | Path,
) -> PersistedArtifactRecord:
    spec = artifact_spec_for_key(rec.artifact_key)
    local_path = Path(local_path)

    if not fe_cfg.persistence.prune_local_after_remote_push:
        rec.notes.append("Local pruning disabled by config.")
        return rec

    if not spec.prune_local_after_push:
        rec.notes.append("Artifact spec does not allow local prune.")
        return rec

    if fe_cfg.persistence.verify_remote_before_prune:
        if not remote_artifact_exists(rec.rel_path):
            rec.notes.append("Remote artifact not verified; skipping prune.")
            return rec

    if local_path.exists():
        local_path.unlink()
        rec.pruned_local = True
        rec.exists_local = False
        rec.notes.append("Local artifact pruned after verified remote push.")
        LOGGER.info("PRUNED LOCAL ARTIFACT | rel_path=%s", rec.rel_path)

    return rec

In [111]:
def persist_json_artifact(
    payload: dict[str, Any],
    *,
    artifact_key: str,
    site_id: str,
    config_sig: str | None = None,
    source_name: str | None = None,
    family_name: str | None = None,
    chunk_id: str | None = None,
) -> PersistedArtifactRecord:
    config_sig = config_sig or current_fe_config_signature()
    rel_path = render_artifact_rel_path(
        artifact_key,
        site_id=site_id,
        config_signature=config_sig,
        source_name=source_name,
        family_name=family_name,
        chunk_id=chunk_id,
    )
    local_path = local_artifact_abs_path(rel_path)
    write_json(local_path, payload)

    rec = push_artifact_if_needed(local_path, artifact_key=artifact_key, rel_path=rel_path)
    rec = prune_local_artifact_if_allowed(rec, local_path=local_path)
    return rec


def persist_npz_artifact(
    arrays: dict[str, np.ndarray],
    *,
    artifact_key: str,
    site_id: str,
    config_sig: str | None = None,
    source_name: str,
    family_name: str,
    chunk_id: str,
) -> PersistedArtifactRecord:
    config_sig = config_sig or current_fe_config_signature()
    rel_path = render_artifact_rel_path(
        artifact_key,
        site_id=site_id,
        config_signature=config_sig,
        source_name=source_name,
        family_name=family_name,
        chunk_id=chunk_id,
    )
    local_path = local_artifact_abs_path(rel_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(local_path, **arrays)

    rec = push_artifact_if_needed(local_path, artifact_key=artifact_key, rel_path=rel_path)
    rec = prune_local_artifact_if_allowed(rec, local_path=local_path)
    return rec


def try_load_json_artifact(
    *,
    artifact_key: str,
    site_id: str,
    config_sig: str | None = None,
    source_name: str | None = None,
    family_name: str | None = None,
    chunk_id: str | None = None,
) -> dict[str, Any] | None:
    config_sig = config_sig or current_fe_config_signature()
    rel_path = render_artifact_rel_path(
        artifact_key,
        site_id=site_id,
        config_signature=config_sig,
        source_name=source_name,
        family_name=family_name,
        chunk_id=chunk_id,
    )
    local_path = local_artifact_abs_path(rel_path)

    if local_path.exists():
        LOGGER.info("LOAD ARTIFACT (local) | key=%s | rel_path=%s", artifact_key, rel_path)
        return read_json(local_path)

    if remote_artifact_exists(rel_path):
        LOGGER.info("LOAD ARTIFACT (remote) | key=%s | rel_path=%s", artifact_key, rel_path)
        pulled = artifact_store.pull(rel_path, local_path=local_path)
        return read_json(pulled)

    return None

## Generic family and chunk registries

In [112]:
@dataclass(frozen=True)
class FeatureFamilySpec:
    key: str
    source_name: str
    runtime_tier: RuntimeTier
    required_halo_px: int
    representation_target: RepresentationTarget
    outputs_dense_layers: bool = True
    outputs_object_fields: bool = False
    notes: str = ""


NAIP_FAMILY_SPECS = {
    "raw": FeatureFamilySpec(
        key="raw",
        source_name="naip",
        runtime_tier=RuntimeTier.CHEAP,
        required_halo_px=0,
        representation_target=RepresentationTarget.RASTER,
        notes="Raw NAIP channels.",
    ),
    "veg_idx": FeatureFamilySpec(
        key="veg_idx",
        source_name="naip",
        runtime_tier=RuntimeTier.CHEAP,
        required_halo_px=0,
        representation_target=RepresentationTarget.RASTER,
        notes="Vegetation indices like NDVI / VARI.",
    ),
    "texture": FeatureFamilySpec(
        key="texture",
        source_name="naip",
        runtime_tier=RuntimeTier.MODERATE,
        required_halo_px=max(fe_cfg.naip.texture_entropy_window, max(fe_cfg.naip.texture_blur_sizes, default=0)),
        representation_target=RepresentationTarget.RASTER,
        notes="Entropy / LBP / blur / local structure features.",
    ),
    "multiscale": FeatureFamilySpec(
        key="multiscale",
        source_name="naip",
        runtime_tier=RuntimeTier.MODERATE,
        required_halo_px=max(fe_cfg.naip.multiscale_sizes, default=0),
        representation_target=RepresentationTarget.RASTER,
        notes="Multi-neighborhood statistics.",
    ),
    "color_diff": FeatureFamilySpec(
        key="color_diff",
        source_name="naip",
        runtime_tier=RuntimeTier.MODERATE,
        required_halo_px=1,
        representation_target=RepresentationTarget.RASTER,
    ),
    "granulometry": FeatureFamilySpec(
        key="granulometry",
        source_name="naip",
        runtime_tier=RuntimeTier.EXPENSIVE,
        required_halo_px=max(fe_cfg.naip.granulometry_scales, default=0),
        representation_target=RepresentationTarget.RASTER,
    ),
    "fractal": FeatureFamilySpec(
        key="fractal",
        source_name="naip",
        runtime_tier=RuntimeTier.EXPENSIVE,
        required_halo_px=max(fe_cfg.naip.fractal_scales, default=0),
        representation_target=RepresentationTarget.RASTER,
    ),
    "wavelet": FeatureFamilySpec(
        key="wavelet",
        source_name="naip",
        runtime_tier=RuntimeTier.EXPENSIVE,
        required_halo_px=16,
        representation_target=RepresentationTarget.RASTER,
    ),
    "gabor": FeatureFamilySpec(
        key="gabor",
        source_name="naip",
        runtime_tier=RuntimeTier.EXPENSIVE,
        required_halo_px=32,
        representation_target=RepresentationTarget.RASTER,
    ),
    "ldp": FeatureFamilySpec(
        key="ldp",
        source_name="naip",
        runtime_tier=RuntimeTier.MODERATE,
        required_halo_px=1,
        representation_target=RepresentationTarget.RASTER,
    ),
    "morphology": FeatureFamilySpec(
        key="morphology",
        source_name="naip",
        runtime_tier=RuntimeTier.EXPENSIVE,
        required_halo_px=max(fe_cfg.naip.morphology_window_sizes, default=0),
        representation_target=RepresentationTarget.RASTER,
    ),
}

In [113]:
def family_spec_frame(specs: dict[str, FeatureFamilySpec]) -> pd.DataFrame:
    rows = []
    for key, spec in specs.items():
        rows.append(
            {
                "family": key,
                "source_name": spec.source_name,
                "runtime_tier": spec.runtime_tier.value,
                "required_halo_px": spec.required_halo_px,
                "target": spec.representation_target.value,
                "notes": spec.notes,
            }
        )
    return pd.DataFrame(rows).sort_values("family").reset_index(drop=True)


display(family_spec_frame(NAIP_FAMILY_SPECS))

,family,source_name,runtime_tier,required_halo_px,target,notes
0,color_diff,naip,moderate,1,raster,
1,fractal,naip,expensive,9,raster,
2,gabor,naip,expensive,32,raster,
3,granulometry,naip,expensive,11,raster,
4,ldp,naip,moderate,1,raster,
5,morphology,naip,expensive,11,raster,
6,multiscale,naip,moderate,15,raster,Multi-neighborhood statistics.
7,raw,naip,cheap,0,raster,Raw NAIP channels.
8,texture,naip,moderate,11,raster,Entropy / LBP / blur / local structure features.
9,veg_idx,naip,cheap,0,raster,Vegetation indices like NDVI / VARI.


In [114]:
def reproject_band_to_grid(
    src_band: np.ndarray,
    *,
    src_transform: Affine,
    src_crs: Any,
    dst_grid: CanonicalGrid,
    resampling: Resampling = Resampling.bilinear,
    dst_nodata: float = np.nan,
) -> np.ndarray:
    dst = np.full((dst_grid.height, dst_grid.width), dst_nodata, dtype=np.float32)

    reproject(
        source=src_band.astype(np.float32),
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_grid.transform,
        dst_crs=dst_grid.crs,
        src_nodata=np.nan,
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    return dst


def align_bundle_to_grid(
    bundle: SourceRasterBundle,
    *,
    dst_grid: CanonicalGrid,
    resampling: Resampling,
) -> SourceRasterBundle:
    aligned = {}
    for name, band in bundle.arrays.items():
        aligned[name] = reproject_band_to_grid(
            band,
            src_transform=bundle.transform,
            src_crs=bundle.crs,
            dst_grid=dst_grid,
            resampling=resampling,
        )

    return SourceRasterBundle(
        site_id=bundle.site_id,
        source_name=bundle.source_name,
        arrays=aligned,
        transform=dst_grid.transform,
        crs=dst_grid.crs,
        nodata=np.nan,
        metadata={**bundle.metadata, "aligned_to": dst_grid.source_name},
    )

In [115]:
def slice_chunk_from_arrays(
    arrays: dict[str, np.ndarray],
    record: ChunkRecord,
    *,
    full_height: int,
    full_width: int,
    halo_px: int | None = None,
) -> tuple[dict[str, np.ndarray], dict[str, int]]:
    bounds = expanded_chunk_bounds(record, height=full_height, width=full_width, halo_px=halo_px)
    out = {}
    for name, arr in arrays.items():
        out[name] = arr[
            bounds["row_start"]:bounds["row_end"],
            bounds["col_start"]:bounds["col_end"]
        ]
    return out, bounds


def crop_chunk_interior(
    expanded_arrays: dict[str, np.ndarray],
    record: ChunkRecord,
    expanded_bounds: dict[str, int],
) -> dict[str, np.ndarray]:
    row0 = record.row_start - expanded_bounds["row_start"]
    row1 = row0 + record.height
    col0 = record.col_start - expanded_bounds["col_start"]
    col1 = col0 + record.width

    cropped = {}
    for name, arr in expanded_arrays.items():
        cropped[name] = arr[row0:row1, col0:col1]
    return cropped

## NAIP

In [116]:
def nan_to_num_copy(arr: np.ndarray, fill_value: float = 0.0) -> np.ndarray:
    out = arr.astype(np.float32, copy=True)
    out[~np.isfinite(out)] = fill_value
    return out


def safe_divide(num: np.ndarray, denom: np.ndarray, fill_value: float = 0.0) -> np.ndarray:
    out = np.full_like(num, fill_value, dtype=np.float32)
    mask = np.isfinite(num) & np.isfinite(denom) & (np.abs(denom) > 1e-12)
    out[mask] = (num[mask] / denom[mask]).astype(np.float32)
    return out


def quantize_image(arr: np.ndarray, levels: int = 256) -> np.ndarray:
    arr = nan_to_num_copy(arr)
    vmin = np.nanmin(arr)
    vmax = np.nanmax(arr)
    if abs(vmax - vmin) < 1e-12:
        return np.zeros_like(arr, dtype=np.uint8)
    scaled = (arr - vmin) / (vmax - vmin)
    scaled = np.clip(scaled, 0, 1)
    return (scaled * (levels - 1)).astype(np.uint8)


def gradient_magnitude(arr: np.ndarray) -> np.ndarray:
    arr = nan_to_num_copy(arr)
    gy, gx = np.gradient(arr)
    return np.sqrt(gx**2 + gy**2).astype(np.float32)


def local_std(arr: np.ndarray, size: int = 5) -> np.ndarray:
    arr = nan_to_num_copy(arr)
    mean = get_uniform_blur(arr, neighborhood_size=size)
    mean_sq = get_uniform_blur(arr**2, neighborhood_size=size)
    var = np.maximum(mean_sq - mean**2, 0.0)
    return np.sqrt(var).astype(np.float32)


def choose_base_band(arrays: dict[str, np.ndarray]) -> tuple[str, np.ndarray]:
    for name in fe_cfg.naip.base_band_preference:
        if name in arrays:
            return name, arrays[name]
    first = next(iter(arrays.keys()))
    return first, arrays[first]


def get_simple_color_diff_kernels() -> dict[str, np.ndarray]:
    return {
        "sobel_x_like": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),
        "sobel_y_like": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32),
        "laplacian_like": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32),
    }

In [117]:
def active_naip_family_names() -> tuple[str, ...]:
    enabled = set(fe_cfg.naip.enabled_families) | set(fe_cfg.naip.heavy_families_enabled)
    ordered = []
    for key in NAIP_FAMILY_SPECS:
        if key in enabled:
            ordered.append(key)
    return tuple(ordered)


def compute_naip_family_on_chunk(
    family_name: str,
    arrays: dict[str, np.ndarray],
) -> dict[str, np.ndarray]:
    out = {}

    if family_name == "raw":
        for name, arr in arrays.items():
            out[f"naip_raw_{name}"] = arr.astype(np.float32)
        return out

    if family_name == "veg_idx":
        red = arrays.get("red")
        green = arrays.get("green")
        blue = arrays.get("blue")
        nir = arrays.get("nir")
        if red is not None and nir is not None:
            out["naip_idx_ndvi"] = safe_divide(nir - red, nir + red)
        if red is not None and green is not None and blue is not None:
            out["naip_idx_vari"] = safe_divide(green - red, green + red - blue)
        if red is not None and green is not None:
            out["naip_idx_g_minus_r"] = (green - red).astype(np.float32)
        if green is not None and blue is not None:
            out["naip_idx_g_minus_b"] = (green - blue).astype(np.float32)
        return out

    base_name, base = choose_base_band(arrays)
    base_clean = nan_to_num_copy(base)
    base_tex = quantize_image(base_clean, levels=fe_cfg.naip.quantize_levels) if fe_cfg.naip.quantize_for_texture else base_clean

    if family_name == "texture":
        out["naip_tex_gradmag"] = gradient_magnitude(base_clean)
        out["naip_tex_localstd_5"] = local_std(base_clean, size=5)
        out["naip_tex_entropy"] = get_entropy_feature(
            base_tex,
            neighborhood_size=fe_cfg.naip.texture_entropy_window
        ).astype(np.float32)
        out[f"naip_tex_lbp_r{fe_cfg.naip.texture_lbp_radius}"] = get_fast_lbp_texture(
            base_tex, radius=fe_cfg.naip.texture_lbp_radius
        ).astype(np.float32)
        for size in fe_cfg.naip.texture_blur_sizes:
            out[f"naip_tex_blur_{size}"] = get_uniform_blur(base_clean, neighborhood_size=size).astype(np.float32)
        return out

    if family_name == "multiscale":
        selected = [name for name in ["red", "green", "nir"] if name in arrays]
        if not selected:
            selected = [base_name]
        for band_name in selected:
            band = nan_to_num_copy(arrays[band_name])
            for size in fe_cfg.naip.multiscale_sizes:
                out[f"naip_ms_mean_{band_name}_{size}"] = get_uniform_blur(band, neighborhood_size=size).astype(np.float32)
                out[f"naip_ms_std_{band_name}_{size}"] = local_std(band, size=size)
        return out

    if family_name == "color_diff":
        for kernel_name, kernel in get_simple_color_diff_kernels().items():
            out[f"naip_cd_{kernel_name}"] = get_color_diff(base_clean, kernel).astype(np.float32)
        return out

    if family_name == "granulometry":
        cube = get_granulometry_features(base_clean, scales=list(fe_cfg.naip.granulometry_scales))
        for i in range(cube.shape[2]):
            out[f"naip_gran_{i:02d}"] = cube[:, :, i].astype(np.float32)
        return out

    if family_name == "fractal":
        out["naip_frac_fd"] = get_fractal_dimension_map(base_clean, scales=list(fe_cfg.naip.fractal_scales)).astype(np.float32)
        return out

    if family_name == "wavelet":
        cube = get_wavelet_features(
            base_clean,
            wavelet_type=fe_cfg.naip.wavelet_type,
            level=fe_cfg.naip.wavelet_level,
        )
        for i in range(cube.shape[2]):
            out[f"naip_wav_{i:02d}"] = cube[:, :, i].astype(np.float32)
        return out

    if family_name == "gabor":
        idx = 0
        for freq in fe_cfg.naip.gabor_frequencies:
            for theta in fe_cfg.naip.gabor_thetas:
                out[f"naip_gabor_{idx:02d}"] = fast_cpu_gabor(base_clean, frequency=freq, theta=theta).astype(np.float32)
                idx += 1
        return out

    if family_name == "ldp":
        out["naip_ldp"] = get_ldp_feature(base_tex, k=fe_cfg.naip.ldp_k).astype(np.float32)
        return out

    if family_name == "morphology":
        for size in fe_cfg.naip.morphology_window_sizes:
            opened = grey_opening(base_clean, size=(size, size)).astype(np.float32)
            closed = grey_closing(base_clean, size=(size, size)).astype(np.float32)
            out[f"naip_morph_open_{size}"] = opened
            out[f"naip_morph_close_{size}"] = closed
            out[f"naip_morph_open_resid_{size}"] = (base_clean - opened).astype(np.float32)
            out[f"naip_morph_close_resid_{size}"] = (closed - base_clean).astype(np.float32)
        return out

    raise KeyError(f"Unsupported NAIP family: {family_name}")

In [118]:
def family_chunk_stage_cache_dir(site_id: str, source_name: str, family_name: str, chunk_id: str, data_sig: str, config_sig: str) -> Path:
    return stage_cache_dir(
        stage_name="family_compute",
        site_id=site_id,
        source_name=f"{source_name}__{family_name}__{chunk_id}",
        data_signature=data_sig,
        config_signature=config_sig,
    )


def family_chunk_signatures(
    *,
    site_id: str,
    source_name: str,
    family_name: str,
    chunk_record: ChunkRecord,
    band_names: Iterable[str],
) -> tuple[str, str]:
    data_sig = hash_payload(
        {
            "site_id": site_id,
            "source_name": source_name,
            "family_name": family_name,
            "chunk_id": chunk_record.chunk_id,
            "chunk_bounds": asdict(chunk_record),
            "band_names": sorted(list(band_names)),
        }
    )
    config_sig = hash_payload(
        {
            "family_name": family_name,
            "naip_config": asdict(fe_cfg.naip),
            "version": fe_cfg.version,
        }
    )
    return data_sig, config_sig

In [119]:
def load_npz_dict(path: str | Path) -> dict[str, np.ndarray]:
    data = np.load(path)
    return {k: data[k] for k in data.files}


def run_naip_family_on_chunk_with_cache(
    *,
    site_id: str,
    family_name: str,
    aligned_bundle: SourceRasterBundle,
    chunk_record: ChunkRecord,
    config_sig: str,
) -> tuple[dict[str, np.ndarray], PersistedArtifactRecord]:
    family_spec = NAIP_FAMILY_SPECS[family_name]
    chunk_arrays_expanded, expanded_bounds = slice_chunk_from_arrays(
        aligned_bundle.arrays,
        chunk_record,
        full_height=aligned_bundle.shape[0],
        full_width=aligned_bundle.shape[1],
        halo_px=family_spec.required_halo_px,
    )

    data_sig, family_cfg_sig = family_chunk_signatures(
        site_id=site_id,
        source_name="naip",
        family_name=family_name,
        chunk_record=chunk_record,
        band_names=chunk_arrays_expanded.keys(),
    )
    cache_dir = family_chunk_stage_cache_dir(
        site_id,
        "naip",
        family_name,
        chunk_record.chunk_id,
        data_sig,
        family_cfg_sig,
    )

    rel_path = render_artifact_rel_path(
        "family_chunk_npz",
        site_id=site_id,
        config_signature=config_sig,
        source_name="naip",
        family_name=family_name,
        chunk_id=chunk_record.chunk_id,
    )
    local_npz = local_artifact_abs_path(rel_path)

    def build_cache_record() -> PersistedArtifactRecord:
        remote_exists = remote_artifact_exists(rel_path)
        remote_ref = None
        # If the local file exists but remote_ref is unknown, we at least preserve the fact
        # that the artifact is remotely addressable by rel_path.
        if remote_exists:
            remote_ref = rel_path
        return PersistedArtifactRecord(
            artifact_key="family_chunk_npz",
            rel_path=rel_path,
            local_path=str(local_npz),
            remote_ref=remote_ref,
            storage_tier=FE_ARTIFACT_SPECS["family_chunk_npz"].storage_tier.value,
            exists_local=local_npz.exists(),
            exists_remote=remote_exists,
            pruned_local=(not local_npz.exists()) and remote_exists,
            notes=["Recovered persistence metadata from cache-hit path."],
        )

    if is_valid_stage_cache(
        stage_cache_dir=cache_dir,
        expected_stage_name="family_compute",
        expected_data_signature=data_sig,
        expected_config_signature=family_cfg_sig,
    ):
        if local_npz.exists():
            LOGGER.info(
                "FAMILY CHUNK CACHE HIT (local) | site=%s | family=%s | chunk=%s",
                site_id, family_name, chunk_record.chunk_id
            )
            return load_npz_dict(local_npz), build_cache_record()

        if remote_artifact_exists(rel_path):
            LOGGER.info(
                "FAMILY CHUNK CACHE HIT (remote) | site=%s | family=%s | chunk=%s",
                site_id, family_name, chunk_record.chunk_id
            )
            pulled = artifact_store.pull(rel_path, local_path=local_npz)
            rec = build_cache_record()
            rec.exists_local = True
            rec.local_path = str(pulled)
            return load_npz_dict(pulled), rec

    with timed_stage(
        "family_chunk_compute",
        site_id=site_id,
        source_name="naip",
        family_name=f"{family_name}:{chunk_record.chunk_id}",
        unit_amount=array_megapixels(next(iter(chunk_arrays_expanded.values()))),
    ):
        expanded_result = compute_naip_family_on_chunk(family_name, chunk_arrays_expanded)
        interior_result = crop_chunk_interior(expanded_result, chunk_record, expanded_bounds)

        persist_rec = persist_npz_artifact(
            interior_result,
            artifact_key="family_chunk_npz",
            site_id=site_id,
            config_sig=config_sig,
            source_name="naip",
            family_name=family_name,
            chunk_id=chunk_record.chunk_id,
        )

        write_stage_cache_manifest(
            stage_cache_dir=cache_dir,
            stage_name="family_compute",
            data_signature=data_sig,
            config_signature=family_cfg_sig,
            module_variants=[],
            artifact_paths={
                "family_chunk_rel_path": rel_path,
                "persisted_local_path": persist_rec.local_path,
                "persisted_remote_ref": persist_rec.remote_ref,
            },
            success=True,
            notes=[f"Computed family chunk for {family_name} / {chunk_record.chunk_id}"],
        )

        return interior_result, persist_rec

In [120]:
def stack_registry_cache_dir(site_id: str, config_sig: str) -> Path:
    return stage_cache_dir(
        stage_name="stack_finalize",
        site_id=site_id,
        source_name="stack_registry",
        data_signature=site_id,
        config_signature=config_sig,
    )


def _layer_registry_key(layer: RasterLayerRecord) -> tuple:
    return (
        layer.site_id,
        layer.source_name,
        layer.family_name,
        layer.layer_name,
        layer.rel_path,
    )


def deduplicate_stack_registry(registry: RasterStackRegistry) -> RasterStackRegistry:
    seen = set()
    deduped = []
    for layer in registry.layers:
        key = _layer_registry_key(layer)
        if key not in seen:
            seen.add(key)
            deduped.append(layer)
    registry.layers = deduped
    return registry


def load_or_init_stack_registry(site_id: str, config_sig: str) -> RasterStackRegistry:
    cache_dir = stack_registry_cache_dir(site_id, config_sig)
    path = cache_dir / "stack_registry.json"

    if path.exists():
        payload = read_json(path)
        layers = [RasterLayerRecord(**row) for row in payload["layers"]]
        reg = RasterStackRegistry(site_id=payload["site_id"], config_signature=payload["config_signature"], layers=layers)
        return deduplicate_stack_registry(reg)

    return RasterStackRegistry(site_id=site_id, config_signature=config_sig, layers=[])


def save_stack_registry(registry: RasterStackRegistry) -> PersistedArtifactRecord:
    registry = deduplicate_stack_registry(registry)

    cache_dir = stack_registry_cache_dir(registry.site_id, registry.config_signature)
    path = cache_dir / "stack_registry.json"

    write_json(
        path,
        {
            "site_id": registry.site_id,
            "config_signature": registry.config_signature,
            "layers": [asdict(layer) for layer in registry.layers],
        },
    )

    rel_path = render_artifact_rel_path(
        "stack_registry",
        site_id=registry.site_id,
        config_signature=registry.config_signature,
    )
    final_local = local_artifact_abs_path(rel_path)
    final_local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, final_local)

    rec = push_artifact_if_needed(final_local, artifact_key="stack_registry", rel_path=rel_path)
    return rec


def register_family_chunk_outputs(
    registry: RasterStackRegistry,
    *,
    family_name: str,
    chunk_record: ChunkRecord,
    chunk_arrays: dict[str, np.ndarray],
    persist_rec: PersistedArtifactRecord,
):
    existing = {_layer_registry_key(layer) for layer in registry.layers}

    for layer_name, arr in chunk_arrays.items():
        layer = RasterLayerRecord(
            site_id=registry.site_id,
            source_name="naip",
            family_name=family_name,
            layer_name=f"{layer_name}::{chunk_record.chunk_id}",
            rel_path=persist_rec.rel_path,
            local_path=persist_rec.local_path,
            remote_ref=persist_rec.remote_ref,
            chunked=True,
            dtype=str(arr.dtype),
            shape=tuple(arr.shape),
            notes=[chunk_record.chunk_id] + list(persist_rec.notes),
        )
        key = _layer_registry_key(layer)
        if key not in existing:
            registry.layers.append(layer)
            existing.add(key)

In [124]:
def source_family_stage_cache_dir(
    site_id: str,
    source_name: str,
    family_name: str,
    chunk_id: str,
    data_sig: str,
    config_sig: str,
) -> Path:
    return stage_cache_dir(
        stage_name="family_compute",
        site_id=site_id,
        source_name=f"{source_name}__{family_name}__{chunk_id}",
        data_signature=data_sig,
        config_signature=config_sig,
    )


def source_family_chunk_signatures(
    *,
    site_id: str,
    source_name: str,
    family_name: str,
    chunk_record: ChunkRecord,
    band_names: Iterable[str],
    family_cfg_payload: dict[str, Any] | None = None,
) -> tuple[str, str]:
    data_sig = hash_payload(
        {
            "site_id": site_id,
            "source_name": source_name,
            "family_name": family_name,
            "chunk_id": chunk_record.chunk_id,
            "chunk_bounds": asdict(chunk_record),
            "band_names": sorted(list(band_names)),
        }
    )
    config_sig = hash_payload(
        {
            "source_name": source_name,
            "family_name": family_name,
            "family_cfg_payload": family_cfg_payload or {},
            "version": fe_cfg.version,
        }
    )
    return data_sig, config_sig


def register_source_family_chunk_outputs(
    registry: RasterStackRegistry,
    *,
    source_name: str,
    family_name: str,
    chunk_record: ChunkRecord,
    chunk_arrays: dict[str, np.ndarray],
    persist_rec: PersistedArtifactRecord,
):
    existing = {
        (
            layer.site_id,
            layer.source_name,
            layer.family_name,
            layer.layer_name,
            layer.rel_path,
        )
        for layer in registry.layers
    }

    for layer_name, arr in chunk_arrays.items():
        layer = RasterLayerRecord(
            site_id=registry.site_id,
            source_name=source_name,
            family_name=family_name,
            layer_name=f"{layer_name}::{chunk_record.chunk_id}",
            rel_path=persist_rec.rel_path,
            local_path=persist_rec.local_path,
            remote_ref=persist_rec.remote_ref,
            chunked=True,
            dtype=str(arr.dtype),
            shape=tuple(arr.shape),
            notes=[chunk_record.chunk_id] + list(persist_rec.notes),
        )
        key = (
            layer.site_id,
            layer.source_name,
            layer.family_name,
            layer.layer_name,
            layer.rel_path,
        )
        if key not in existing:
            registry.layers.append(layer)
            existing.add(key)


def run_source_family_on_chunk_with_cache(
    *,
    site_id: str,
    source_name: str,
    family_name: str,
    aligned_bundle: SourceRasterBundle,
    chunk_record: ChunkRecord,
    family_spec: FeatureFamilySpec,
    family_compute_fn,
    family_cfg_payload: dict[str, Any] | None = None,
    config_sig: str | None = None,
) -> tuple[dict[str, np.ndarray], PersistedArtifactRecord]:
    config_sig = config_sig or fe_config_signature()

    chunk_arrays_expanded, expanded_bounds = slice_chunk_from_arrays(
        aligned_bundle.arrays,
        chunk_record,
        full_height=aligned_bundle.shape[0],
        full_width=aligned_bundle.shape[1],
        halo_px=family_spec.required_halo_px,
    )

    data_sig, family_cfg_sig = source_family_chunk_signatures(
        site_id=site_id,
        source_name=source_name,
        family_name=family_name,
        chunk_record=chunk_record,
        band_names=chunk_arrays_expanded.keys(),
        family_cfg_payload=family_cfg_payload,
    )

    cache_dir = source_family_stage_cache_dir(
        site_id=site_id,
        source_name=source_name,
        family_name=family_name,
        chunk_id=chunk_record.chunk_id,
        data_sig=data_sig,
        config_sig=family_cfg_sig,
    )

    rel_path = render_artifact_rel_path(
        "family_chunk_npz",
        site_id=site_id,
        config_signature=config_sig,
        source_name=source_name,
        family_name=family_name,
        chunk_id=chunk_record.chunk_id,
    )
    local_npz = local_artifact_abs_path(rel_path)

    def build_cache_record() -> PersistedArtifactRecord:
        remote_exists = remote_artifact_exists(rel_path)
        remote_ref = rel_path if remote_exists else None
        return PersistedArtifactRecord(
            artifact_key="family_chunk_npz",
            rel_path=rel_path,
            local_path=str(local_npz),
            remote_ref=remote_ref,
            storage_tier=FE_ARTIFACT_SPECS["family_chunk_npz"].storage_tier.value,
            exists_local=local_npz.exists(),
            exists_remote=remote_exists,
            pruned_local=(not local_npz.exists()) and remote_exists,
            notes=[f"Recovered persistence metadata from cache-hit path ({source_name}/{family_name})."],
        )

    if is_valid_stage_cache(
        stage_cache_dir=cache_dir,
        expected_stage_name="family_compute",
        expected_data_signature=data_sig,
        expected_config_signature=family_cfg_sig,
    ):
        if local_npz.exists():
            LOGGER.info(
                "FAMILY CHUNK CACHE HIT (local) | site=%s | source=%s | family=%s | chunk=%s",
                site_id, source_name, family_name, chunk_record.chunk_id
            )
            return load_npz_dict(local_npz), build_cache_record()

        if remote_artifact_exists(rel_path):
            LOGGER.info(
                "FAMILY CHUNK CACHE HIT (remote) | site=%s | source=%s | family=%s | chunk=%s",
                site_id, source_name, family_name, chunk_record.chunk_id
            )
            pulled = artifact_store.pull(rel_path, local_path=local_npz)
            rec = build_cache_record()
            rec.exists_local = True
            rec.local_path = str(pulled)
            return load_npz_dict(pulled), rec

    with timed_stage(
        "family_chunk_compute",
        site_id=site_id,
        source_name=source_name,
        family_name=f"{family_name}:{chunk_record.chunk_id}",
        unit_amount=array_megapixels(next(iter(chunk_arrays_expanded.values()))),
    ):
        expanded_result = family_compute_fn(chunk_arrays_expanded)
        interior_result = crop_chunk_interior(expanded_result, chunk_record, expanded_bounds)

        persist_rec = persist_npz_artifact(
            interior_result,
            artifact_key="family_chunk_npz",
            site_id=site_id,
            config_sig=config_sig,
            source_name=source_name,
            family_name=family_name,
            chunk_id=chunk_record.chunk_id,
        )

        write_stage_cache_manifest(
            stage_cache_dir=cache_dir,
            stage_name="family_compute",
            data_signature=data_sig,
            config_signature=family_cfg_sig,
            module_variants=[],
            artifact_paths={
                "family_chunk_rel_path": rel_path,
                "persisted_local_path": persist_rec.local_path,
                "persisted_remote_ref": persist_rec.remote_ref,
            },
            success=True,
            notes=[f"Computed family chunk for {source_name}/{family_name}/{chunk_record.chunk_id}"],
        )

        return interior_result, persist_rec


def run_source_chunked_pipeline(
    *,
    site_id: str,
    source_name: str,
    raw_bundle: SourceRasterBundle,
    canonical_grid: CanonicalGrid,
    chunk_manifest: ChunkManifest,
    family_specs: dict[str, FeatureFamilySpec],
    family_compute_fns: dict[str, Any],
    family_cfg_payloads: dict[str, dict[str, Any]] | None = None,
    registry: RasterStackRegistry | None = None,
    rebuild_registry: bool = False,
) -> RasterStackRegistry:
    config_sig = fe_config_signature()

    aligned_bundle = align_bundle_to_grid(
        raw_bundle,
        dst_grid=canonical_grid,
        resampling=SOURCE_SPECS[source_name].default_resampling,
    )

    if registry is None or rebuild_registry:
        registry = RasterStackRegistry(site_id=site_id, config_signature=config_sig, layers=[])

    LOGGER.info(
        "SOURCE CHUNKED RUN | site=%s | source=%s | families=%s | n_chunks=%d",
        site_id, source_name, tuple(family_specs.keys()), len(chunk_manifest.records)
    )

    for family_name, family_spec in family_specs.items():
        compute_fn = family_compute_fns[family_name]
        cfg_payload = (family_cfg_payloads or {}).get(family_name, {})

        LOGGER.info(
            "SOURCE FAMILY START | site=%s | source=%s | family=%s",
            site_id, source_name, family_name
        )

        for idx, record in enumerate(chunk_manifest.records, start=1):
            est_remaining = estimate_remaining_time(
                "family_chunk_compute",
                remaining_units=max(len(chunk_manifest.records) - idx, 0),
            )
            if est_remaining is not None:
                LOGGER.info(
                    "SOURCE ETA | site=%s | source=%s | family=%s | chunk=%s | est_remaining_sec=%.2f",
                    site_id, source_name, family_name, record.chunk_id, est_remaining
                )

            chunk_arrays, persist_rec = run_source_family_on_chunk_with_cache(
                site_id=site_id,
                source_name=source_name,
                family_name=family_name,
                aligned_bundle=aligned_bundle,
                chunk_record=record,
                family_spec=family_spec,
                family_compute_fn=compute_fn,
                family_cfg_payload=cfg_payload,
                config_sig=config_sig,
            )

            register_source_family_chunk_outputs(
                registry,
                source_name=source_name,
                family_name=family_name,
                chunk_record=record,
                chunk_arrays=chunk_arrays,
                persist_rec=persist_rec,
            )

        LOGGER.info(
            "SOURCE FAMILY DONE | site=%s | source=%s | family=%s",
            site_id, source_name, family_name
        )

    registry = deduplicate_stack_registry(registry)
    save_stack_registry(registry)
    LOGGER.info(
        "SOURCE CHUNKED RUN COMPLETE | site=%s | source=%s | registered_layers=%d",
        site_id, source_name, len(registry.layers)
    )
    return registry

In [125]:
def run_naip_chunked_pipeline(
    site_id: str,
    *,
    assets: SiteAssetBundle | None = None,
    grid: CanonicalGrid | None = None,
    chunk_manifest: ChunkManifest | None = None,
    rebuild_registry: bool = True,
) -> RasterStackRegistry:
    config_sig = fe_config_signature()

    if assets is None:
        assets = prepare_site_assets(site_id, force_refresh=fe_cfg.force_refresh_assets)
    if grid is None:
        grid = build_canonical_grid_for_site(site_id, assets=assets, force_refresh=fe_cfg.force_refresh_features)
    if chunk_manifest is None:
        chunk_manifest = build_chunk_manifest(grid, force_refresh=fe_cfg.force_rebuild_chunk_manifest)

    naip_path = assets.source_assets["naip"]
    raw_bundle = read_raster_bundle(
        naip_path,
        site_id=site_id,
        source_name="naip",
        band_names=infer_naip_band_names(naip_path),
    )
    aligned_bundle = align_bundle_to_grid(
        raw_bundle,
        dst_grid=grid,
        resampling=SOURCE_SPECS["naip"].default_resampling,
    )

    family_names = active_naip_family_names()
    registry = RasterStackRegistry(site_id=site_id, config_signature=config_sig, layers=[]) if rebuild_registry else load_or_init_stack_registry(site_id, config_sig)

    LOGGER.info(
        "NAIP CHUNKED RUN | site=%s | families=%s | n_chunks=%d",
        site_id, family_names, len(chunk_manifest.records)
    )

    for family_name in family_names:
        LOGGER.info("NAIP FAMILY START | site=%s | family=%s", site_id, family_name)

        for idx, record in enumerate(chunk_manifest.records, start=1):
            est_remaining = estimate_remaining_time(
                "family_chunk_compute",
                remaining_units=max(len(chunk_manifest.records) - idx, 0),
            )
            if est_remaining is not None:
                LOGGER.info(
                    "NAIP ETA | site=%s | family=%s | chunk=%s | est_remaining_sec=%.2f",
                    site_id, family_name, record.chunk_id, est_remaining
                )

            chunk_arrays, persist_rec = run_naip_family_on_chunk_with_cache(
                site_id=site_id,
                family_name=family_name,
                aligned_bundle=aligned_bundle,
                chunk_record=record,
                config_sig=config_sig,
            )

            register_family_chunk_outputs(
                registry,
                family_name=family_name,
                chunk_record=record,
                chunk_arrays=chunk_arrays,
                persist_rec=persist_rec,
            )

        LOGGER.info("NAIP FAMILY DONE | site=%s | family=%s", site_id, family_name)

    registry = deduplicate_stack_registry(registry)
    save_stack_registry(registry)
    LOGGER.info("NAIP CHUNKED RUN COMPLETE | site=%s | registered_layers=%d", site_id, len(registry.layers))
    return registry

In [126]:
def refresh_artifact_store():
    global artifact_store
    artifact_store = build_artifact_store(fe_cfg.storage)
    LOGGER.info("Refreshed artifact store | type=%s", type(artifact_store).__name__)
    return artifact_store

In [127]:
fe_cfg.storage.enable_drive_store = True
fe_cfg.storage.use_hybrid_store = True
fe_cfg.storage.enable_local_store = True

fe_cfg.persistence.push_large_artifacts_to_remote = True
fe_cfg.persistence.prune_local_after_remote_push = True
fe_cfg.persistence.verify_remote_before_prune = True

artifact_store = refresh_artifact_store()
print(type(artifact_store).__name__)

2026-04-17 03:56:19,514 | INFO     | features.notebook | Using HybridArtifactStore
2026-04-17 03:56:19,514 | INFO     | features.notebook | Refreshed artifact store | type=HybridArtifactStore


HybridArtifactStore


In [128]:
TEST_SITE = fe_cfg.sites[0]

naip_stack_registry = run_naip_chunked_pipeline(
    TEST_SITE,
    assets=site_assets,
    grid=canonical_grid,
    chunk_manifest=chunk_manifest,
    rebuild_registry=True,
)

print("Registered layers:", len(naip_stack_registry.layers))
display(pd.DataFrame([asdict(x) for x in naip_stack_registry.layers[:10]]))

2026-04-17 03:56:33,494 | INFO     | features.notebook | NAIP CHUNKED RUN | site=calaveras-big-trees | families=('raw', 'veg_idx', 'texture', 'multiscale') | n_chunks=48
2026-04-17 03:56:33,494 | INFO     | features.notebook | NAIP FAMILY START | site=calaveras-big-trees | family=raw
2026-04-17 03:56:33,594 | INFO     | features.notebook | FAMILY CHUNK CACHE HIT (local) | site=calaveras-big-trees | family=raw | chunk=chunk_00000


[artifact_store] Using cached Google Drive credentials.


2026-04-17 03:56:36,524 | INFO     | features.notebook | FAMILY CHUNK CACHE HIT (local) | site=calaveras-big-trees | family=raw | chunk=chunk_00001
2026-04-17 03:56:38,593 | INFO     | features.notebook | FAMILY CHUNK CACHE HIT (local) | site=calaveras-big-trees | family=raw | chunk=chunk_00002
2026-04-17 03:56:40,689 | INFO     | features.notebook | FAMILY CHUNK CACHE HIT (local) | site=calaveras-big-trees | family=raw | chunk=chunk_00003
2026-04-17 03:56:42,770 | INFO     | features.notebook | FAMILY CHUNK CACHE HIT (local) | site=calaveras-big-trees | family=raw | chunk=chunk_00004
2026-04-17 03:56:45,040 | INFO     | features.notebook | FAMILY CHUNK CACHE HIT (local) | site=calaveras-big-trees | family=raw | chunk=chunk_00005
2026-04-17 03:56:47,352 | INFO     | features.notebook | FAMILY CHUNK CACHE HIT (local) | site=calaveras-big-trees | family=raw | chunk=chunk_00006
2026-04-17 03:56:49,777 | INFO     | features.notebook | FAMILY CHUNK CACHE HIT (local) | site=calaveras-big-tre

[artifact_store] Updating existing Drive artifact for features/calaveras-big-trees/91fef1517fca02cc/stack/stack_registry.json


2026-04-17 03:59:25,720 | INFO     | features.notebook | NAIP CHUNKED RUN COMPLETE | site=calaveras-big-trees | registered_layers=1536


Registered layers: 1536


,site_id,source_name,family_name,layer_name,rel_path,local_path,remote_ref,chunked,dtype,shape,notes
0,calaveras-big-trees,naip,raw,naip_raw_red::chunk_00000,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00000, Recovered persistence metadata f..."
1,calaveras-big-trees,naip,raw,naip_raw_green::chunk_00000,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00000, Recovered persistence metadata f..."
2,calaveras-big-trees,naip,raw,naip_raw_blue::chunk_00000,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00000, Recovered persistence metadata f..."
3,calaveras-big-trees,naip,raw,naip_raw_nir::chunk_00000,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00000, Recovered persistence metadata f..."
4,calaveras-big-trees,naip,raw,naip_raw_red::chunk_00001,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00001, Recovered persistence metadata f..."
5,calaveras-big-trees,naip,raw,naip_raw_green::chunk_00001,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00001, Recovered persistence metadata f..."
6,calaveras-big-trees,naip,raw,naip_raw_blue::chunk_00001,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00001, Recovered persistence metadata f..."
7,calaveras-big-trees,naip,raw,naip_raw_nir::chunk_00001,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00001, Recovered persistence metadata f..."
8,calaveras-big-trees,naip,raw,naip_raw_red::chunk_00002,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00002, Recovered persistence metadata f..."
9,calaveras-big-trees,naip,raw,naip_raw_green::chunk_00002,features/calaveras-big-trees/91fef1517fca02cc/...,/home/jovyan/work/Dry-shRub/shrub/Final/artifa...,features/calaveras-big-trees/91fef1517fca02cc/...,True,float32,"(1024, 1024)","[chunk_00002, Recovered persistence metadata f..."


# Object aggregation and stack assembly

In [129]:
def pixel_from_xy(transform: Affine, x: float, y: float) -> tuple[int, int]:
    col, row = ~transform * (x, y)
    return int(round(row)), int(round(col))


def clip_window(row: int, col: int, radius: int, height: int, width: int) -> tuple[slice, slice]:
    r0 = max(0, row - radius)
    r1 = min(height, row + radius + 1)
    c0 = max(0, col - radius)
    c1 = min(width, col + radius + 1)
    return slice(r0, r1), slice(c0, c1)


def summarize_patch(arr: np.ndarray, rs: slice, cs: slice, stats: tuple[str, ...]) -> dict[str, float]:
    patch = arr[rs, cs]
    vals = patch[np.isfinite(patch)]
    if vals.size == 0:
        return {stat: np.nan for stat in stats}

    out = {}
    if "mean" in stats:
        out["mean"] = float(np.mean(vals))
    if "std" in stats:
        out["std"] = float(np.std(vals))
    if "min" in stats:
        out["min"] = float(np.min(vals))
    if "max" in stats:
        out["max"] = float(np.max(vals))
    return out

In [130]:
def stack_registry_frame(registry: RasterStackRegistry) -> pd.DataFrame:
    return pd.DataFrame([asdict(x) for x in registry.layers]) if registry.layers else pd.DataFrame()


def parse_chunked_layer_name(layer_name: str) -> tuple[str, str]:
    if "::" not in layer_name:
        raise ValueError(f"Expected chunked layer name with '::', got: {layer_name}")
    base_name, chunk_id = layer_name.split("::", 1)
    return base_name, chunk_id


def selected_registry_rows(
    registry: RasterStackRegistry,
    *,
    source_name: str | None = None,
    family_names: Iterable[str] | None = None,
    base_layer_names: Iterable[str] | None = None,
) -> pd.DataFrame:
    df = stack_registry_frame(registry)
    if df.empty:
        return df

    if source_name is not None:
        df = df[df["source_name"] == source_name].copy()

    if family_names is not None:
        family_names = set(family_names)
        df = df[df["family_name"].isin(family_names)].copy()

    if base_layer_names is not None:
        wanted = set(base_layer_names)
        df["base_layer_name"] = df["layer_name"].str.split("::").str[0]
        df = df[df["base_layer_name"].isin(wanted)].copy()

    return df.reset_index(drop=True)

In [131]:
def load_chunk_layer_array(row: pd.Series) -> np.ndarray:
    layer_name = row["layer_name"]
    base_name, _ = parse_chunked_layer_name(layer_name)

    local_path = row.get("local_path")
    rel_path = row.get("rel_path")

    path = Path(local_path) if local_path is not None else None

    if path is not None and path.exists():
        payload = load_npz_dict(path)
        if base_name not in payload:
            raise KeyError(f"Base layer {base_name} not present in chunk npz at {path}. Keys={list(payload.keys())}")
        return payload[base_name]

    if rel_path is not None and remote_artifact_exists(rel_path):
        LOGGER.info("PULL CHUNK ARTIFACT | rel_path=%s", rel_path)
        target_local = local_artifact_abs_path(rel_path)
        target_local.parent.mkdir(parents=True, exist_ok=True)
        pulled = artifact_store.pull(rel_path, local_path=target_local)

        payload = load_npz_dict(pulled)
        if base_name not in payload:
            raise KeyError(f"Base layer {base_name} not present in pulled chunk npz at {pulled}. Keys={list(payload.keys())}")
        return payload[base_name]

    raise FileNotFoundError(
        f"Chunk artifact unavailable for layer={layer_name}. "
        f"local_path={local_path}, rel_path={rel_path}, remote_ref={row.get('remote_ref')}"
    )


def assemble_full_layer_from_registry(
    registry: RasterStackRegistry,
    chunk_manifest: ChunkManifest,
    canonical_grid: CanonicalGrid,
    *,
    target_layer_name: str,
) -> np.ndarray:
    rows = selected_registry_rows(registry, base_layer_names=[target_layer_name])
    if rows.empty:
        raise ValueError(f"No registered chunks found for target layer {target_layer_name}")

    out = np.full((canonical_grid.height, canonical_grid.width), np.nan, dtype=np.float32)
    chunk_lookup = {r.chunk_id: r for r in chunk_manifest.records}

    with timed_stage(
        "stack_assemble",
        site_id=canonical_grid.site_id,
        source_name="assembled_stack",
        family_name=target_layer_name,
        unit_amount=(canonical_grid.height * canonical_grid.width) / 1_000_000.0,
    ):
        for _, row in rows.iterrows():
            _, chunk_id = parse_chunked_layer_name(row["layer_name"])
            record = chunk_lookup[chunk_id]
            arr = load_chunk_layer_array(row)

            if arr.shape != (record.height, record.width):
                raise ValueError(
                    f"Chunk array shape mismatch for {chunk_id}: expected {(record.height, record.width)}, got {arr.shape}"
                )

            out[record.row_start:record.row_end, record.col_start:record.col_end] = arr

    return out

In [132]:
def assemble_selected_layers(
    registry: RasterStackRegistry,
    chunk_manifest: ChunkManifest,
    canonical_grid: CanonicalGrid,
    *,
    source_name: str | None = None,
    family_names: Iterable[str] | None = None,
    base_layer_names: Iterable[str] | None = None,
) -> dict[str, np.ndarray]:
    rows = selected_registry_rows(
        registry,
        source_name=source_name,
        family_names=family_names,
        base_layer_names=base_layer_names,
    )
    if rows.empty:
        return {}

    rows = rows.copy()
    rows["base_layer_name"] = rows["layer_name"].str.split("::").str[0]

    arrays = {}
    for base_layer_name in sorted(rows["base_layer_name"].unique()):
        arrays[base_layer_name] = assemble_full_layer_from_registry(
            registry,
            chunk_manifest,
            canonical_grid,
            target_layer_name=base_layer_name,
        )
    return arrays

In [133]:
def aggregate_selected_registered_layers_to_objects(
    registry: RasterStackRegistry,
    *,
    site_id: str,
    canonical_grid: CanonicalGrid,
    chunk_manifest: ChunkManifest,
    objects_df: pd.DataFrame,
    source_name: str | None = None,
    family_names: Iterable[str] | None = None,
    base_layer_names: Iterable[str] | None = None,
) -> pd.DataFrame:
    if objects_df.empty:
        LOGGER.info("OBJECT AGG | site=%s | no objects provided", site_id)
        return pd.DataFrame()

    layers = assemble_selected_layers(
        registry,
        chunk_manifest,
        canonical_grid,
        source_name=source_name,
        family_names=family_names,
        base_layer_names=base_layer_names,
    )
    if not layers:
        LOGGER.warning("OBJECT AGG | site=%s | no assembled layers selected", site_id)
        return pd.DataFrame()

    work = objects_df.copy()

    x_col = "x_naip" if "x_naip" in work.columns else ("x_als" if "x_als" in work.columns else None)
    y_col = "y_naip" if "y_naip" in work.columns else ("y_als" if "y_als" in work.columns else None)
    if x_col is None or y_col is None:
        raise ValueError("Could not find object coordinates for aggregation.")

    rows = []
    h, w = canonical_grid.height, canonical_grid.width
    px_size = canonical_grid.pixel_size[0]

    with timed_stage(
        "object_aggregation",
        site_id=site_id,
        source_name=source_name or "selected_layers",
        unit_amount=len(work),
        extra={"n_layers": len(layers)},
    ):
        for _, obj in work.iterrows():
            x = float(obj[x_col])
            y = float(obj[y_col])
            row, col = pixel_from_xy(canonical_grid.transform, x, y)

            base = {
                "site_id": site_id,
                "object_id": obj.get("object_id"),
                "plot_id": obj.get("plot_id"),
                "row": row,
                "col": col,
            }

            radius_px = fe_cfg.object_agg.square_window_radius_px
            if fe_cfg.object_agg.use_radius_scaled_window and "radius_m" in obj and pd.notna(obj["radius_m"]):
                radius_px = int(round(float(obj["radius_m"]) / px_size))
                radius_px = max(fe_cfg.object_agg.min_radius_px, min(fe_cfg.object_agg.max_radius_px, radius_px))

            if row < 0 or row >= h or col < 0 or col >= w:
                base["valid_sample"] = False
                rows.append(base)
                continue

            base["valid_sample"] = True
            rs, cs = clip_window(row, col, radius_px, h, w)

            for feat_name, arr in layers.items():
                if fe_cfg.object_agg.include_centroid_sample:
                    base[f"{feat_name}__centroid"] = float(arr[row, col]) if np.isfinite(arr[row, col]) else np.nan

                if not fe_cfg.object_agg.centroid_only:
                    stats = summarize_patch(arr, rs, cs, fe_cfg.object_agg.stats)
                    for stat_name, value in stats.items():
                        base[f"{feat_name}__{stat_name}"] = value

            rows.append(base)

    out_df = pd.DataFrame(rows)
    LOGGER.info(
        "OBJECT AGG DONE | site=%s | n_objects=%d | n_rows=%d | n_cols=%d",
        site_id, len(work), len(out_df), out_df.shape[1]
    )
    return out_df

In [134]:
def persist_object_feature_table(
    obj_df: pd.DataFrame,
    *,
    site_id: str,
    config_sig: str | None = None,
) -> PersistedArtifactRecord:
    config_sig = config_sig or current_fe_config_signature()
    rel_path = render_artifact_rel_path(
        "object_feature_table",
        site_id=site_id,
        config_signature=config_sig,
    )
    local_path = local_artifact_abs_path(rel_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    obj_df.to_csv(local_path, index=False)

    rec = push_artifact_if_needed(local_path, artifact_key="object_feature_table", rel_path=rel_path)
    return rec


def build_and_persist_object_feature_table(
    registry: RasterStackRegistry,
    *,
    site_id: str,
    canonical_grid: CanonicalGrid,
    chunk_manifest: ChunkManifest,
    objects_df: pd.DataFrame,
    source_name: str | None = None,
    family_names: Iterable[str] | None = None,
    base_layer_names: Iterable[str] | None = None,
) -> tuple[pd.DataFrame, PersistedArtifactRecord | None]:
    obj_df = aggregate_selected_registered_layers_to_objects(
        registry,
        site_id=site_id,
        canonical_grid=canonical_grid,
        chunk_manifest=chunk_manifest,
        objects_df=objects_df,
        source_name=source_name,
        family_names=family_names,
        base_layer_names=base_layer_names,
    )

    if obj_df.empty:
        LOGGER.warning("No object features produced for site=%s; skipping persistence.", site_id)
        return obj_df, None

    rec = persist_object_feature_table(obj_df, site_id=site_id)
    LOGGER.info("Persisted object feature table | site=%s | path=%s", site_id, rec.local_path)
    return obj_df, rec

In [135]:
def export_single_band_geotiff(
    arr: np.ndarray,
    *,
    canonical_grid: CanonicalGrid,
    out_path: str | Path,
    nodata: float = np.nan,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    profile = canonical_grid.profile(dtype="float32", count=1, nodata=nodata)
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(arr.astype(np.float32), 1)


def export_selected_layers_as_geotiffs(
    assembled_layers: dict[str, np.ndarray],
    *,
    canonical_grid: CanonicalGrid,
    site_id: str,
    export_root: str | Path | None = None,
) -> dict[str, Path]:
    export_root = Path(export_root) if export_root is not None else ensure_dir(fe_cfg.summary_root / site_id / "selected_stack_exports")
    export_root.mkdir(parents=True, exist_ok=True)

    out_paths = {}
    for layer_name, arr in assembled_layers.items():
        out_path = export_root / f"{layer_name}.tif"
        export_single_band_geotiff(arr, canonical_grid=canonical_grid, out_path=out_path)
        out_paths[layer_name] = out_path

    LOGGER.info("Exported %d selected layers as GeoTIFFs | site=%s | root=%s", len(out_paths), site_id, export_root)
    return out_paths


def export_selected_layers_as_npz(
    assembled_layers: dict[str, np.ndarray],
    *,
    site_id: str,
    export_root: str | Path | None = None,
) -> Path:
    export_root = Path(export_root) if export_root is not None else ensure_dir(fe_cfg.summary_root / site_id / "selected_stack_exports")
    export_root.mkdir(parents=True, exist_ok=True)

    out_path = export_root / "selected_layers.npz"
    np.savez_compressed(out_path, **assembled_layers)
    LOGGER.info("Exported selected layers as NPZ | site=%s | path=%s", site_id, out_path)
    return out_path

# 3DEP

In [137]:
TERRAIN_FAMILY_SPECS = {
    "terrain": FeatureFamilySpec(
        key="terrain",
        source_name="3dep",
        runtime_tier=RuntimeTier.MODERATE,
        required_halo_px=16,
        representation_target=RepresentationTarget.RASTER,
        notes="Elevation, slope, aspect, northness/eastness, curvature, ruggedness, TPI, exposure.",
    ),
}
display(family_spec_frame(TERRAIN_FAMILY_SPECS))

,family,source_name,runtime_tier,required_halo_px,target,notes
0,terrain,3dep,moderate,16,raster,"Elevation, slope, aspect, northness/eastness, ..."


In [136]:
def make_terrain_family_compute_fn(
    *,
    pixel_size_m: float,
    ruggedness_kernel_radius: int,
    tpi_radius_meters: float,
    hillshade_azimuth: float,
    hillshade_zenith: float,
):
    def _compute(arrays: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
        elevation = nan_to_num_copy(next(iter(arrays.values())))

        gy, gx = np.gradient(elevation)
        slope = np.sqrt(gx**2 + gy**2).astype(np.float32)

        aspect = np.arctan2(gy, gx).astype(np.float32)
        northness = np.cos(aspect).astype(np.float32)
        eastness = np.sin(aspect).astype(np.float32)

        gyy, gyx = np.gradient(gy)
        gxy, gxx = np.gradient(gx)
        curvature = (gxx + gyy).astype(np.float32)

        ruggedness = local_std(
            elevation,
            size=max(3, 2 * ruggedness_kernel_radius + 1),
        )

        tpi_window = max(5, int(round(tpi_radius_meters / max(pixel_size_m, 1e-6))))
        local_mean = get_uniform_blur(elevation, neighborhood_size=tpi_window)
        tpi = (elevation - local_mean).astype(np.float32)

        az = math.radians(hillshade_azimuth)
        zen = math.radians(hillshade_zenith)
        hillshade = (
            np.cos(zen) * np.cos(slope) +
            np.sin(zen) * np.sin(slope) * np.cos(az - aspect)
        ).astype(np.float32)

        return {
            "terrain_elevation": elevation.astype(np.float32),
            "terrain_slope": slope,
            "terrain_aspect": aspect,
            "terrain_northness": northness,
            "terrain_eastness": eastness,
            "terrain_curvature": curvature,
            "terrain_ruggedness": ruggedness,
            "terrain_tpi": tpi,
            "terrain_hillshade_exposure": hillshade,
        }

    return _compute

In [142]:
def site_3dep_cache_root(site_id: str) -> Path:
    root = fe_output_root() / "site_assets" / site_id / "3dep"
    root.mkdir(parents=True, exist_ok=True)
    return root


def expected_3dep_filename(site_id: str) -> str:
    slug = site_id.replace("-", "_")
    return f"{slug}_3dep.tif"


def render_3dep_asset_rel_path(site_id: str, filename: str | None = None) -> str:
    filename = filename or expected_3dep_filename(site_id)
    return f"features/{site_id}/shared/site_assets/3dep/{filename}"


def local_3dep_asset_path(site_id: str, filename: str | None = None) -> Path:
    filename = filename or expected_3dep_filename(site_id)
    return site_3dep_cache_root(site_id) / filename


def validate_cached_3dep(tif_path: str | Path) -> bool:
    tif_path = Path(tif_path)
    if not tif_path.exists():
        return False

    try:
        with rasterio.open(tif_path) as src:
            if src.count < 1 or src.width <= 0 or src.height <= 0:
                return False

            # tiny probe read
            w = min(16, src.width)
            h = min(16, src.height)
            _ = src.read(1, window=rasterio.windows.Window(0, 0, w, h))
        return True
    except Exception as e:
        LOGGER.warning("Cached 3DEP validation failed for %s: %s", tif_path, e)
        return False


def persist_site_3dep_asset(site_id: str, local_path: str | Path) -> PersistedArtifactRecord:
    local_path = Path(local_path)
    rel_path = render_3dep_asset_rel_path(site_id, filename=local_path.name)
    remote_ref = artifact_store.push(local_path, rel_path=rel_path)

    return PersistedArtifactRecord(
        artifact_key="site_3dep_raster",
        rel_path=rel_path,
        local_path=str(local_path),
        remote_ref=remote_ref,
        storage_tier="shared",
        exists_local=local_path.exists(),
        exists_remote=True,
        pruned_local=False,
        notes=[f"Persisted 3DEP asset for site={site_id}"],
    )

In [143]:
def build_3dep_source_inventory_record(
    site_id: str,
    *,
    site_assets: SiteAssetBundle,
) -> SourceIngestRecord:
    source_assets = dict(site_assets.source_assets or {})

    if "3dep" in source_assets and source_assets["3dep"]:
        return SourceIngestRecord(
            site_id=site_id,
            source_name="3dep",
            asset_path=str(source_assets["3dep"]),
            status="registered",
            notes=["3DEP asset already registered in site_assets.source_assets"],
        )

    notes = []
    candidate_keys = ["3dep", "dem", "terrain", "naip_3dep_product"]
    for key in candidate_keys:
        if key in source_assets and source_assets[key]:
            return SourceIngestRecord(
                site_id=site_id,
                source_name="3dep",
                asset_path=str(source_assets[key]),
                status="registered_alias",
                notes=[f"Using source_assets[{key!r}] as 3DEP raster"],
            )

    return SourceIngestRecord(
        site_id=site_id,
        source_name="3dep",
        asset_path=None,
        status="missing",
        notes=["No 3DEP asset currently registered in site_assets.source_assets"],
    )

In [145]:
@dataclass
class SourceIngestRecord:
    site_id: str
    source_name: str
    asset_path: str | None = None
    status: str = "unknown"
    notes: list[str] = field(default_factory=list)


def prepare_3dep_asset(
    site_id: str,
    *,
    site_assets: SiteAssetBundle,
    force_refresh: bool = False,
) -> Path:
    record = build_3dep_source_inventory_record(site_id, site_assets=site_assets)
    rel_path = render_3dep_asset_rel_path(site_id)

    local_path = local_3dep_asset_path(site_id)

    # 1) existing local cache
    if not force_refresh and validate_cached_3dep(local_path):
        LOGGER.info("Using cached local 3DEP asset | site=%s | path=%s", site_id, local_path)
        return local_path

    # 2) hydrate from artifact store
    if not force_refresh and artifact_store.exists(rel_path):
        LOGGER.info("Hydrating 3DEP asset from artifact store | site=%s | rel=%s", site_id, rel_path)
        hydrated = artifact_store.pull(rel_path, local_path=local_path)
        if validate_cached_3dep(hydrated):
            return hydrated
        LOGGER.warning("Hydrated 3DEP asset failed validation | site=%s | path=%s", site_id, hydrated)

    # 3) use registered site asset path and persist into canonical cache/store
    if record.asset_path is None:
        raise FileNotFoundError(
            f"No 3DEP asset is registered for site={site_id}. "
            f"Expected it in site_assets.source_assets['3dep'] or a recognized alias."
        )

    src_path = Path(record.asset_path)
    if not src_path.exists():
        raise FileNotFoundError(f"Registered 3DEP asset does not exist for site={site_id}: {src_path}")

    local_path.parent.mkdir(parents=True, exist_ok=True)
    if src_path.resolve() != local_path.resolve():
        shutil.copy2(src_path, local_path)

    if not validate_cached_3dep(local_path):
        raise RuntimeError(f"Prepared 3DEP raster is invalid for site={site_id}: {local_path}")

    persist_site_3dep_asset(site_id, local_path)

    LOGGER.info(
        "Prepared 3DEP asset from registered source | site=%s | src=%s | local=%s",
        site_id, src_path, local_path
    )
    return local_path


def load_3dep_source_bundle(
    site_id: str,
    *,
    site_assets: SiteAssetBundle,
    force_refresh: bool = False,
) -> SourceRasterBundle:
    tif_path = prepare_3dep_asset(
        site_id,
        site_assets=site_assets,
        force_refresh=force_refresh,
    )
    return read_raster_bundle(
        tif_path,
        site_id=site_id,
        source_name="3dep",
        band_names=["elevation"],
    )


def align_3dep_to_canonical_grid(
    bundle: SourceRasterBundle,
    *,
    canonical_grid: CanonicalGrid,
) -> SourceRasterBundle:
    return align_bundle_to_grid(
        bundle,
        dst_grid=canonical_grid,
        resampling=SOURCE_SPECS["3dep"].default_resampling,
    )


In [146]:
def run_3dep_chunked_pipeline(
    site_id: str,
    *,
    site_assets: SiteAssetBundle,
    canonical_grid: CanonicalGrid,
    chunk_manifest: ChunkManifest,
    registry: RasterStackRegistry | None = None,
    force_refresh_source: bool = False,
) -> RasterStackRegistry:
    raw_bundle = load_3dep_source_bundle(
        site_id,
        site_assets=site_assets,
        force_refresh=force_refresh_source,
    )

    terrain_compute_fn = make_terrain_family_compute_fn(
        pixel_size_m=canonical_grid.pixel_size[0],
        ruggedness_kernel_radius=fe_cfg.terrain.ruggedness_kernel_radius,
        tpi_radius_meters=fe_cfg.terrain.tpi_radius_meters,
        hillshade_azimuth=fe_cfg.terrain.hillshade_azimuth,
        hillshade_zenith=fe_cfg.terrain.hillshade_zenith,
    )

    return run_source_chunked_pipeline(
        site_id=site_id,
        source_name="3dep",
        raw_bundle=raw_bundle,
        canonical_grid=canonical_grid,
        chunk_manifest=chunk_manifest,
        family_specs=TERRAIN_FAMILY_SPECS,
        family_compute_fns={
            "terrain": terrain_compute_fn,
        },
        family_cfg_payloads={
            "terrain": {
                "ruggedness_kernel_radius": fe_cfg.terrain.ruggedness_kernel_radius,
                "tpi_radius_meters": fe_cfg.terrain.tpi_radius_meters,
                "hillshade_azimuth": fe_cfg.terrain.hillshade_azimuth,
                "hillshade_zenith": fe_cfg.terrain.hillshade_zenith,
                "pixel_size_m": canonical_grid.pixel_size[0],
            }
        },
        registry=registry,
    )

In [ ]:
TEST_SITE = fe_cfg.sites[0]

registry_3dep = run_3dep_chunked_pipeline(
    TEST_SITE,
    site_assets=site_assets,
    canonical_grid=canonical_grid,
    chunk_manifest=chunk_manifest,
    registry=RasterStackRegistry(
        site_id=TEST_SITE,
        config_signature=fe_config_signature(),
        layers=[],
    ),
)

display(stack_registry_frame(registry_3dep).head(30))
print(f"3DEP layers registered: {len(registry_3dep.layers)}")

In [ ]:
def run_3dep_all_sites(
    *,
    force_refresh_assets: bool = False,
    force_refresh_source: bool = False,
) -> dict[str, RasterStackRegistry]:
    out = {}

    for site_id in fe_cfg.sites:
        LOGGER.info("=" * 80)
        LOGGER.info("RUN 3DEP ALL SITES | site=%s", site_id)

        site_assets = prepare_site_assets(
            site_id,
            force_refresh=force_refresh_assets,
        )

        canonical_grid = build_canonical_grid_for_site(
            site_id,
            assets=site_assets,
            force_refresh=fe_cfg.force_refresh_features,
        )

        chunk_manifest = build_chunk_manifest(
            canonical_grid,
            force_refresh=fe_cfg.force_rebuild_chunk_manifest,
        )

        registry = run_3dep_chunked_pipeline(
            site_id,
            site_assets=site_assets,
            canonical_grid=canonical_grid,
            chunk_manifest=chunk_manifest,
            registry=RasterStackRegistry(
                site_id=site_id,
                config_signature=fe_config_signature(),
                layers=[],
            ),
            force_refresh_source=force_refresh_source,
        )

        out[site_id] = registry

    return out

In [ ]:
rows = []
for site_id in fe_cfg.sites:
    site_assets = prepare_site_assets(site_id, force_refresh=False)
    rec = build_3dep_source_inventory_record(site_id, site_assets=site_assets)
    rows.append({
        "site_id": site_id,
        "status": rec.status,
        "asset_path": rec.asset_path,
        "notes": " | ".join(rec.notes),
    })

display(pd.DataFrame(rows))

In [ ]:
all_3dep_runs = run_3dep_all_sites(
    force_refresh_assets=False,
    force_refresh_source=False,
)

summary_rows = []
for site_id, registry in all_3dep_runs.items():
    df = stack_registry_frame(registry)
    summary_rows.append({
        "site_id": site_id,
        "n_layers": len(registry.layers),
        "families": sorted(df["family_name"].unique().tolist()) if not df.empty else [],
        "sources": sorted(df["source_name"].unique().tolist()) if not df.empty else [],
    })

display(pd.DataFrame(summary_rows))

# RAP

In [140]:
RAP_FAMILY_SPECS = {
    "prior": FeatureFamilySpec(
        key="prior",
        source_name="rap",
        runtime_tier=RuntimeTier.CHEAP,
        required_halo_px=8,
        representation_target=RepresentationTarget.RASTER,
        notes="Local shrub presence and neighborhood ecological priors.",
    ),
}
display(family_spec_frame(RAP_FAMILY_SPECS))

,family,source_name,runtime_tier,required_halo_px,target,notes
0,prior,rap,cheap,8,raster,Local shrub presence and neighborhood ecologic...


In [72]:
def prepare_rap_asset_from_local_path(
    site_id: str,
    *,
    tif_path: str | Path,
) -> Path:
    tif_path = Path(tif_path)
    if not tif_path.exists():
        raise FileNotFoundError(f"RAP tif not found: {tif_path}")

    local_path = site_rap_cache_root(site_id) / tif_path.name
    if local_path.resolve() != tif_path.resolve():
        shutil.copy2(tif_path, local_path)

    LOGGER.info("Prepared RAP local asset | site=%s | path=%s", site_id, local_path)
    return local_path


def infer_rap_band_names(path: str | Path) -> list[str]:
    path = Path(path)
    with rasterio.open(path) as src:
        count = src.count

    if count >= 4:
        default = ["SHR", "TRE", "PFG", "AFG"]
        if count > 4:
            default += [f"band_{i}" for i in range(5, count + 1)]
        return default[:count]
    return [f"band_{i}" for i in range(1, count + 1)]


def load_rap_source_bundle(
    site_id: str,
    *,
    tif_path: str | Path,
) -> SourceRasterBundle:
    return read_raster_bundle(
        tif_path,
        site_id=site_id,
        source_name="rap",
        band_names=infer_rap_band_names(tif_path),
    )


def align_rap_to_canonical_grid(
    bundle: SourceRasterBundle,
    *,
    canonical_grid: CanonicalGrid,
) -> SourceRasterBundle:
    return align_bundle_to_grid(
        bundle,
        dst_grid=canonical_grid,
        resampling=SOURCE_SPECS["rap"].default_resampling,
    )

In [141]:
def make_rap_prior_compute_fn(
    *,
    context_radius_meters: float,
    approx_native_resolution_m: float = 10.0,
):
    def _compute(arrays: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
        out = {}

        shrub_name = "SHR" if "SHR" in arrays else next(iter(arrays.keys()))
        shrub = nan_to_num_copy(arrays[shrub_name]).astype(np.float32)

        window = max(5, int(round(context_radius_meters / approx_native_resolution_m)))

        out["rap_local_shrub_presence"] = shrub
        out["rap_shrub_fraction_prior"] = get_uniform_blur(
            shrub,
            neighborhood_size=window,
        ).astype(np.float32)

        comp_names = [name for name in arrays.keys() if name != shrub_name]
        for name in comp_names:
            out[f"rap_{name}_prior"] = get_uniform_blur(
                nan_to_num_copy(arrays[name]),
                neighborhood_size=window,
            ).astype(np.float32)

        return out

    return _compute


def run_rap_chunked_pipeline(
    site_id: str,
    *,
    tif_path: str | Path,
    canonical_grid: CanonicalGrid,
    chunk_manifest: ChunkManifest,
    registry: RasterStackRegistry | None = None,
) -> RasterStackRegistry:
    raw_bundle = load_rap_source_bundle(site_id, tif_path=tif_path)

    prior_compute_fn = make_rap_prior_compute_fn(
        context_radius_meters=fe_cfg.rap.context_radius_meters,
        approx_native_resolution_m=10.0,
    )

    return run_source_chunked_pipeline(
        site_id=site_id,
        source_name="rap",
        raw_bundle=raw_bundle,
        canonical_grid=canonical_grid,
        chunk_manifest=chunk_manifest,
        family_specs=RAP_FAMILY_SPECS,
        family_compute_fns={
            "prior": prior_compute_fn,
        },
        family_cfg_payloads={
            "prior": {
                "context_radius_meters": fe_cfg.rap.context_radius_meters,
                "approx_native_resolution_m": 10.0,
            }
        },
        registry=registry,
    )

# ALS

In [74]:
ALS_WORKFLOW_STAGES = pd.DataFrame(
    [
        {
            "stage": "als_select_inputs",
            "purpose": "Select ALS files/tiles intersecting the site or chunk.",
            "output": "tile inventory / selected ALS paths",
        },
        {
            "stage": "als_prepare_hag",
            "purpose": "Ensure Height Above Ground exists for selected LAS/LAZ tiles.",
            "output": "HAG-enabled LAS/LAZ",
        },
        {
            "stage": "als_structural_rasters",
            "purpose": "Create CHM / mean height / distance-to-canopy / local relief / KNN metrics.",
            "output": "rasterized structural products",
        },
        {
            "stage": "als_align_to_canonical",
            "purpose": "Align ALS-derived rasters to canonical grid/chunks.",
            "output": "chunk-ready structural arrays",
        },
        {
            "stage": "als_family_compute",
            "purpose": "Emit ALS feature families under the same artifact/caching contract.",
            "output": "chunked ALS family outputs",
        },
    ]
)
display(ALS_WORKFLOW_STAGES)

,stage,purpose,output
0,als_select_inputs,Select ALS files/tiles intersecting the site o...,tile inventory / selected ALS paths
1,als_prepare_hag,Ensure Height Above Ground exists for selected...,HAG-enabled LAS/LAZ
2,als_structural_rasters,Create CHM / mean height / distance-to-canopy ...,rasterized structural products
3,als_align_to_canonical,Align ALS-derived rasters to canonical grid/ch...,chunk-ready structural arrays
4,als_family_compute,Emit ALS feature families under the same artif...,chunked ALS family outputs


In [75]:
ALS_EXPECTED_PRODUCTS = pd.DataFrame(
    [
        {"product": "als_chm_max", "source_logic": "create_height_rasters / CHM Max", "target_family": "height"},
        {"product": "als_height_mean", "source_logic": "create_height_rasters / Mean", "target_family": "height"},
        {"product": "als_distance_to_tall_canopy", "source_logic": "calculate_distance_to_tall_canopy", "target_family": "canopy_context"},
        {"product": "als_local_relief", "source_logic": "calculate_local_relief", "target_family": "canopy_context"},
        {"product": "als_knn_variance", "source_logic": "calculate_knn_node_metrics", "target_family": "structure"},
        {"product": "als_knn_roughness", "source_logic": "calculate_knn_node_metrics", "target_family": "structure"},
        {"product": "als_knn_heterogeneity", "source_logic": "calculate_knn_node_metrics", "target_family": "structure"},
        {"product": "als_knn_maxima_density", "source_logic": "calculate_knn_node_metrics", "target_family": "structure"},
    ]
)
display(ALS_EXPECTED_PRODUCTS)

,product,source_logic,target_family
0,als_chm_max,create_height_rasters / CHM Max,height
1,als_height_mean,create_height_rasters / Mean,height
2,als_distance_to_tall_canopy,calculate_distance_to_tall_canopy,canopy_context
3,als_local_relief,calculate_local_relief,canopy_context
4,als_knn_variance,calculate_knn_node_metrics,structure
5,als_knn_roughness,calculate_knn_node_metrics,structure
6,als_knn_heterogeneity,calculate_knn_node_metrics,structure
7,als_knn_maxima_density,calculate_knn_node_metrics,structure


In [76]:
def als_metadata_frame(site_assets: SiteAssetBundle) -> pd.DataFrame:
    rows = site_assets.source_assets.get("als_metadata", [])
    return pd.DataFrame(rows) if rows else pd.DataFrame()


def normalize_als_bounds(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    rename_map = {}
    for src, dst in [
        ("minx", "min_x"),
        ("maxx", "max_x"),
        ("miny", "min_y"),
        ("maxy", "max_y"),
        ("xmin", "min_x"),
        ("xmax", "max_x"),
        ("ymin", "min_y"),
        ("ymax", "max_y"),
    ]:
        if src in out.columns and dst not in out.columns:
            rename_map[src] = dst
    if rename_map:
        out = out.rename(columns=rename_map)

    return out


def canonical_grid_bounds(grid: CanonicalGrid) -> dict[str, float]:
    x_min, y_max = grid.transform * (0, 0)
    x_max, y_min = grid.transform * (grid.width, grid.height)
    return {
        "min_x": min(x_min, x_max),
        "max_x": max(x_min, x_max),
        "min_y": min(y_min, y_max),
        "max_y": max(y_min, y_max),
    }


def bbox_intersects(a: dict[str, float], b: dict[str, float]) -> bool:
    return not (
        a["max_x"] < b["min_x"] or
        a["min_x"] > b["max_x"] or
        a["max_y"] < b["min_y"] or
        a["min_y"] > b["max_y"]
    )


def build_als_tile_inventory(site_assets: SiteAssetBundle, canonical_grid: CanonicalGrid) -> pd.DataFrame:
    df = als_metadata_frame(site_assets)
    if df.empty:
        LOGGER.warning("ALS tile inventory empty.")
        return df

    df = normalize_als_bounds(df)
    grid_bbox = canonical_grid_bounds(canonical_grid)

    if {"min_x", "max_x", "min_y", "max_y"}.issubset(df.columns):
        overlaps = []
        for _, row in df.iterrows():
            tile_bbox = {
                "min_x": float(row["min_x"]),
                "max_x": float(row["max_x"]),
                "min_y": float(row["min_y"]),
                "max_y": float(row["max_y"]),
            }
            overlaps.append(bbox_intersects(tile_bbox, grid_bbox))
        df["intersects_canonical_grid"] = overlaps
    else:
        df["intersects_canonical_grid"] = True

    LOGGER.info("Built ALS tile inventory | rows=%d", len(df))
    return df

In [77]:
def structural_raster_cache_root(site_id: str) -> Path:
    return ensure_dir(fe_cfg.cache_root / "als_structural_rasters" / site_id)


def structural_raster_manifest_path(site_id: str) -> Path:
    return structural_raster_cache_root(site_id) / "structural_raster_manifest.json"


def load_structural_raster_manifest(site_id: str) -> dict[str, Any]:
    path = structural_raster_manifest_path(site_id)
    if path.exists():
        return read_json(path)
    return {"site_id": site_id, "products": []}


def save_structural_raster_manifest(site_id: str, payload: dict[str, Any]) -> Path:
    path = structural_raster_manifest_path(site_id)
    write_json(path, payload)
    LOGGER.info("Saved structural raster manifest | site=%s | path=%s", site_id, path)
    return path

In [ ]:
ALS_FAMILY_SPECS = {
    "height_structure": FeatureFamilySpec(
        key="height_structure",
        source_name="als",
        runtime_tier=RuntimeTier.EXPENSIVE,
        required_halo_px=8,
        representation_target=RepresentationTarget.RASTER,
        notes="ALS-derived structural rasters: canopy height, mean height, density, canopy distance, local relief, and roughness proxies.",
    ),
}
display(family_spec_frame(ALS_FAMILY_SPECS))

In [ ]:
def select_intersecting_als_tiles(
    site_assets: SiteAssetBundle,
    canonical_grid: CanonicalGrid,
) -> pd.DataFrame:
    df = build_als_tile_inventory(site_assets, canonical_grid)
    if df.empty:
        return df

    if "intersects_canonical_grid" in df.columns:
        df = df[df["intersects_canonical_grid"]].copy()

    return df.reset_index(drop=True)


def _safe_height_values_from_las(las) -> np.ndarray:
    if hasattr(las, "HeightAboveGround"):
        return np.asarray(las.HeightAboveGround, dtype=np.float32)
    return np.asarray(las.z, dtype=np.float32)


def rasterize_als_structural_metrics_from_las(
    las_path: str | Path,
    *,
    site_id: str,
    tile_id: str,
    resolution: float = 1.0,
) -> SourceRasterBundle:
    las_path = Path(las_path)
    las = laspy.read(las_path)

    x = np.asarray(las.x, dtype=np.float64)
    y = np.asarray(las.y, dtype=np.float64)
    z = _safe_height_values_from_las(las)

    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x = x[valid]
    y = y[valid]
    z = z[valid]

    if x.size == 0:
        raise ValueError(f"No valid ALS points found in {las_path}")

    x_min, x_max = float(np.min(x)), float(np.max(x))
    y_min, y_max = float(np.min(y)), float(np.max(y))

    cols = max(1, int(np.ceil((x_max - x_min) / resolution)))
    rows = max(1, int(np.ceil((y_max - y_min) / resolution)))

    x_edges = np.linspace(x_min, x_max, cols + 1)
    y_edges = np.linspace(y_min, y_max, rows + 1)

    chm_grid, _, _, _ = binned_statistic_2d(x, y, z, statistic="max", bins=[x_edges, y_edges])
    mean_grid, _, _, _ = binned_statistic_2d(x, y, z, statistic="mean", bins=[x_edges, y_edges])
    count_grid, _, _, _ = binned_statistic_2d(x, y, z, statistic="count", bins=[x_edges, y_edges])
    std_grid, _, _, _ = binned_statistic_2d(x, y, z, statistic="std", bins=[x_edges, y_edges])

    chm_grid = np.rot90(chm_grid).astype(np.float32)
    mean_grid = np.rot90(mean_grid).astype(np.float32)
    count_grid = np.rot90(count_grid).astype(np.float32)
    std_grid = np.rot90(std_grid).astype(np.float32)

    safe_chm = np.nan_to_num(chm_grid, nan=0.0)
    dist_to_canopy = calculate_distance_to_tall_canopy(
        safe_chm,
        resolution=resolution,
        height_threshold=fe_cfg.als.tall_canopy_threshold_m,
    ).astype(np.float32)
    local_relief = calculate_local_relief(
        safe_chm,
        window_size_pixels=fe_cfg.als.local_relief_window_px,
    ).astype(np.float32)

    transform = from_origin(x_min, y_max, resolution, resolution)

    arrays = {
        "als_chm_max": chm_grid,
        "als_height_mean": mean_grid,
        "als_point_count": count_grid,
        "als_height_std": std_grid,
        "als_distance_to_tall_canopy": dist_to_canopy,
        "als_local_relief": local_relief,
    }

    return SourceRasterBundle(
        site_id=site_id,
        source_name="als",
        arrays=arrays,
        transform=transform,
        crs=las.header.parse_crs(),
        nodata=np.nan,
        metadata={
            "tile_id": tile_id,
            "path": str(las_path),
            "resolution": resolution,
            "has_hag": hasattr(las, "HeightAboveGround"),
        },
    )

In [ ]:
def prepare_als_tile_local_copy(
    site_id: str,
    *,
    tile_row: pd.Series,
) -> Path:
    local_dir = structural_raster_cache_root(site_id) / "raw_tiles"
    local_dir.mkdir(parents=True, exist_ok=True)

    name = tile_row.get("name") or tile_row.get("filename") or f"{tile_row.name}.laz"
    local_path = local_dir / str(name)

    if local_path.exists():
        return local_path

    href = tile_row.get("href") or tile_row.get("url") or tile_row.get("download_url")
    if href is None:
        raise ValueError(f"Could not find ALS download URL for tile row: {tile_row.to_dict()}")

    download_file(href, local_path)
    LOGGER.info("Downloaded ALS tile | site=%s | tile=%s | path=%s", site_id, name, local_path)
    return local_path


def align_and_merge_als_tile_bundles(
    bundles: list[SourceRasterBundle],
    *,
    canonical_grid: CanonicalGrid,
) -> SourceRasterBundle:
    if not bundles:
        raise ValueError("No ALS bundles provided for merge.")

    aligned_bundles = [
        align_bundle_to_grid(
            bundle,
            dst_grid=canonical_grid,
            resampling=SOURCE_SPECS["als"].default_resampling,
        )
        for bundle in bundles
    ]

    layer_names = sorted(set().union(*[set(b.arrays.keys()) for b in aligned_bundles]))
    merged = {}

    for layer_name in layer_names:
        stack = []
        for bundle in aligned_bundles:
            if layer_name in bundle.arrays:
                stack.append(bundle.arrays[layer_name])
        if not stack:
            continue

        stacked = np.stack(stack, axis=0)

        if layer_name == "als_point_count":
            merged[layer_name] = np.nansum(stacked, axis=0).astype(np.float32)
        else:
            merged[layer_name] = np.nanmax(stacked, axis=0).astype(np.float32)

    return SourceRasterBundle(
        site_id=canonical_grid.site_id,
        source_name="als",
        arrays=merged,
        transform=canonical_grid.transform,
        crs=canonical_grid.crs,
        nodata=np.nan,
        metadata={
            "n_tiles_merged": len(bundles),
            "aligned_to": canonical_grid.source_name,
        },
    )

In [ ]:
def make_als_height_structure_compute_fn():
    def _compute(arrays: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
        out = {}

        for name, arr in arrays.items():
            out[name] = nan_to_num_copy(arr).astype(np.float32)

        if "als_chm_max" in arrays:
            chm = nan_to_num_copy(arrays["als_chm_max"])
            out["als_chm_gradmag"] = gradient_magnitude(chm)
            out["als_chm_localstd_5"] = local_std(chm, size=5)

        if "als_height_mean" in arrays and "als_chm_max" in arrays:
            out["als_gap_max_minus_mean"] = (
                nan_to_num_copy(arrays["als_chm_max"]) - nan_to_num_copy(arrays["als_height_mean"])
            ).astype(np.float32)

        if "als_point_count" in arrays:
            count = nan_to_num_copy(arrays["als_point_count"])
            out["als_point_density_blur_5"] = get_uniform_blur(count, neighborhood_size=5).astype(np.float32)

        return out

    return _compute


def run_als_chunked_pipeline(
    site_id: str,
    *,
    site_assets: SiteAssetBundle,
    canonical_grid: CanonicalGrid,
    chunk_manifest: ChunkManifest,
    registry: RasterStackRegistry | None = None,
) -> RasterStackRegistry:
    tiles_df = select_intersecting_als_tiles(site_assets, canonical_grid)
    if tiles_df.empty:
        raise ValueError(f"No ALS tiles intersect the canonical grid for site={site_id}")

    tile_bundles = []
    for _, row in tiles_df.iterrows():
        local_tile = prepare_als_tile_local_copy(site_id, tile_row=row)
        tile_id = Path(local_tile).stem
        bundle = rasterize_als_structural_metrics_from_las(
            local_tile,
            site_id=site_id,
            tile_id=tile_id,
            resolution=fe_cfg.als.structural_raster_resolution_m,
        )
        tile_bundles.append(bundle)

    merged_bundle = align_and_merge_als_tile_bundles(
        tile_bundles,
        canonical_grid=canonical_grid,
    )

    return run_source_chunked_pipeline(
        site_id=site_id,
        source_name="als",
        raw_bundle=merged_bundle,
        canonical_grid=canonical_grid,
        chunk_manifest=chunk_manifest,
        family_specs=ALS_FAMILY_SPECS,
        family_compute_fns={
            "height_structure": make_als_height_structure_compute_fn(),
        },
        family_cfg_payloads={
            "height_structure": {
                "structural_raster_resolution_m": fe_cfg.als.structural_raster_resolution_m,
                "tall_canopy_threshold_m": fe_cfg.als.tall_canopy_threshold_m,
                "local_relief_window_px": fe_cfg.als.local_relief_window_px,
            }
        },
        registry=registry,
    )

In [78]:
als_inventory_preview = build_als_tile_inventory(site_assets, canonical_grid)
display(als_inventory_preview.head(10))

structural_manifest_preview = load_structural_raster_manifest(TEST_SITE)
display(pd.DataFrame(structural_manifest_preview.get("products", [])))

2026-04-16 02:38:44,737 | INFO     | features.notebook | Built ALS tile inventory | rows=10


,count,compressed,major_version,minor_version,dataformat_id,srs,native_bounds,srs_wkt,srs_json,stats_keys,metadata_keys,source_file,intersects_canonical_grid
0,36906926,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 739000, 'maxx': 739999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH3935.laz,True
1,36370037,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 740000, 'maxx': 740999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH4035.laz,True
2,39198120,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 740000, 'maxx': 740999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH4036.laz,True
3,39876902,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 741000, 'maxx': 741999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH4136.laz,True
4,37664314,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 741000, 'maxx': 741999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH4137.laz,True
5,32272833,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 742000, 'maxx': 742999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH4236.laz,True
6,37644869,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 742000, 'maxx': 742999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH4237.laz,True
7,39200800,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 742000, 'maxx': 742999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH4238.laz,True
8,32063001,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 743000, 'maxx': 743999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH4337.laz,True
9,36002705,True,1,4,6,"{'compoundwkt': 'COMPD_CS[""NAD83(2011) / UTM z...","{'minx': 743000, 'maxx': 743999.99, 'miny': 42...","COMPD_CS[""NAD83(2011) / UTM zone 10N + NAVD88 ...","{'type': 'CompoundCRS', 'name': 'NAD83(2011) /...","[bbox, statistic]","[comp_spatialreference, compressed, copc, coun...",USGS_LPC_CA_SierraNevada_B22_10SGH4338.laz,True


""


# Multi-source runner

In [ ]:
def run_multisource_fe_notebook_pipeline(
    site_id: str,
    *,
    source_paths: dict[str, str | Path] | None = None,
    include_naip: bool = True,
    include_3dep: bool = False,
    include_rap: bool = False,
    include_als: bool = False,
    rebuild_registry: bool = True,
) -> dict[str, Any]:
    source_paths = source_paths or {}

    site_assets = prepare_site_assets(site_id, force_refresh=fe_cfg.force_refresh_assets)
    canonical_grid = build_canonical_grid_for_site(
        site_id,
        assets=site_assets,
        force_refresh=fe_cfg.force_refresh_features,
    )
    chunk_manifest = build_chunk_manifest(
        canonical_grid,
        force_refresh=fe_cfg.force_rebuild_chunk_manifest,
    )

    registry = RasterStackRegistry(
        site_id=site_id,
        config_signature=fe_config_signature(),
        layers=[],
    ) if rebuild_registry else load_or_init_stack_registry(site_id, fe_config_signature())

    executed_sources = []

    if include_naip:
        registry = run_naip_chunked_pipeline(
            site_id,
            assets=site_assets,
            grid=canonical_grid,
            chunk_manifest=chunk_manifest,
            rebuild_registry=False,
        )
        executed_sources.append("naip")

    if include_3dep:
        registry = run_3dep_chunked_pipeline(
            site_id,
            site_assets=site_assets,
            canonical_grid=canonical_grid,
            chunk_manifest=chunk_manifest,
            registry=registry,
            force_refresh_source=False,
        )
        executed_sources.append("3dep")

    if include_rap:
        tif_path = source_paths.get("rap") or site_assets.source_assets.get("rap")
        if tif_path is None:
            raise ValueError("RAP requested but no tif_path provided and no RAP site asset is registered.")
        registry = run_rap_chunked_pipeline(
            site_id,
            tif_path=tif_path,
            canonical_grid=canonical_grid,
            chunk_manifest=chunk_manifest,
            registry=registry,
        )
        executed_sources.append("rap")

    if include_als:
        registry = run_als_chunked_pipeline(
            site_id,
            site_assets=site_assets,
            canonical_grid=canonical_grid,
            chunk_manifest=chunk_manifest,
            registry=registry,
        )
        executed_sources.append("als")

    registry = deduplicate_stack_registry(registry)
    stack_rec = save_stack_registry(registry)

    return {
        "site_id": site_id,
        "executed_sources": executed_sources,
        "site_assets": site_assets,
        "canonical_grid": canonical_grid,
        "chunk_manifest": chunk_manifest,
        "stack_registry": registry,
        "stack_registry_artifact": stack_rec,
    }

In [ ]:
TEST_SITE = fe_cfg.sites[0]

# Start with NAIP + whichever local external rasters you currently have.
multisource_run = run_multisource_fe_notebook_pipeline(
    TEST_SITE,
    source_paths={
        # "3dep": "/absolute/path/to/site_3dep.tif",
        # "rap": "/absolute/path/to/site_rap.tif",
    },
    include_naip=True,
    include_3dep=False,
    include_rap=False,
    include_als=False,
    rebuild_registry=True,
)

display(stack_registry_frame(multisource_run["stack_registry"]).head(20))
print(multisource_run["executed_sources"])

In [79]:
# Assemble a small subset of NAIP layers back to full-site arrays.
assembled_preview = assemble_selected_layers(
    naip_stack_registry,
    chunk_manifest,
    canonical_grid,
    source_name="naip",
    family_names=["veg_idx"],
    base_layer_names=["naip_idx_ndvi", "naip_idx_vari"],
)

print("Assembled preview layers:", list(assembled_preview.keys()))
display(feature_stack_frame(assembled_preview))

2026-04-16 02:38:44,813 | INFO     | features.notebook | START | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_idx_ndvi
2026-04-16 02:38:46,766 | INFO     | features.notebook | END   | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_idx_ndvi | duration=1.95s | sec_per_unit=0.0491
2026-04-16 02:38:46,801 | INFO     | features.notebook | START | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_idx_vari
2026-04-16 02:38:48,774 | INFO     | features.notebook | END   | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_idx_vari | duration=1.97s | sec_per_unit=0.0496


Assembled preview layers: ['naip_idx_ndvi', 'naip_idx_vari']


,name,shape,finite_fraction,min,max,mean,std
0,naip_idx_ndvi,"(5375, 7398)",1.0,-0.913043,0.832061,0.132904,0.217377
1,naip_idx_vari,"(5375, 7398)",1.0,-15.000000,16.000000,0.169959,0.485313


In [80]:
site_objects = label_objects_for_site(TEST_SITE)

obj_df, obj_rec = build_and_persist_object_feature_table(
    naip_stack_registry,
    site_id=TEST_SITE,
    canonical_grid=canonical_grid,
    chunk_manifest=chunk_manifest,
    objects_df=site_objects,
    source_name="naip",
    family_names=["veg_idx", "texture"],
    base_layer_names=["naip_idx_ndvi", "naip_tex_gradmag"],
)

print(obj_df.shape)
print(obj_rec)

2026-04-16 02:38:49,341 | INFO     | features.notebook | label_objects_for_site | requested=calaveras-big-trees | matched_rows=36 | unique_sites=['calaveras-big-trees', 'dl-bliss', 'independence-lake', 'pacific-union-college', 'sedgwick', 'shaver-lake']
2026-04-16 02:38:49,389 | INFO     | features.notebook | START | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_idx_ndvi
2026-04-16 02:38:51,368 | INFO     | features.notebook | END   | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_idx_ndvi | duration=1.98s | sec_per_unit=0.0497
2026-04-16 02:38:51,402 | INFO     | features.notebook | START | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_tex_gradmag
2026-04-16 02:38:54,691 | INFO     | features.notebook | END   | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_tex_gradmag | duration=3.29s | sec_per_unit=0.0827
2026-04-16 02:38:5

[artifact_store] Creating new Drive artifact for features/calaveras-big-trees/91fef1517fca02cc/objects/object_features.csv


2026-04-16 02:38:58,427 | INFO     | features.notebook | Persisted object feature table | site=calaveras-big-trees | path=/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features/artifact_store_local/features/calaveras-big-trees/91fef1517fca02cc/objects/object_features.csv


(36, 16)
PersistedArtifactRecord(artifact_key='object_feature_table', rel_path='features/calaveras-big-trees/91fef1517fca02cc/objects/object_features.csv', local_path='/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/features/artifact_store_local/features/calaveras-big-trees/91fef1517fca02cc/objects/object_features.csv', remote_ref='1Xz0rVTsowt0nmEvuaho61tIYcMyy-Qz-', storage_tier='local_then_remote', exists_local=True, exists_remote=True, pruned_local=False, notes=[])


In [81]:
naip_object_features_preview = aggregate_selected_registered_layers_to_objects(
    naip_stack_registry,
    site_id=TEST_SITE,
    canonical_grid=canonical_grid,
    chunk_manifest=chunk_manifest,
    objects_df=site_objects,
    source_name="naip",
    family_names=["veg_idx", "texture"],
    base_layer_names=["naip_idx_ndvi", "naip_tex_gradmag"],
)

print("Object feature preview shape:", naip_object_features_preview.shape)
display(naip_object_features_preview.head(5))

2026-04-16 02:38:58,478 | INFO     | features.notebook | START | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_idx_ndvi
2026-04-16 02:39:00,422 | INFO     | features.notebook | END   | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_idx_ndvi | duration=1.94s | sec_per_unit=0.0489
2026-04-16 02:39:00,455 | INFO     | features.notebook | START | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_tex_gradmag
2026-04-16 02:39:03,759 | INFO     | features.notebook | END   | stage=stack_assemble | site=calaveras-big-trees | source=assembled_stack | family=naip_tex_gradmag | duration=3.30s | sec_per_unit=0.0831
2026-04-16 02:39:03,759 | INFO     | features.notebook | START | stage=object_aggregation | site=calaveras-big-trees | source=naip | family=None
2026-04-16 02:39:03,767 | INFO     | features.notebook | END   | stage=object_aggregation | site=calaveras-big-trees | source=n

Object feature preview shape: (36, 16)


,site_id,object_id,plot_id,row,col,valid_sample,naip_idx_ndvi__centroid,naip_idx_ndvi__mean,naip_idx_ndvi__std,naip_idx_ndvi__min,naip_idx_ndvi__max,naip_tex_gradmag__centroid,naip_tex_gradmag__mean,naip_tex_gradmag__std,naip_tex_gradmag__min,naip_tex_gradmag__max
0,calaveras-big-trees,1,CATCU_0009_20250615_1,1142,6601,True,0.333333,0.382695,0.066159,0.270588,0.525773,17.867569,25.453485,10.854493,2.236068,43.176960
1,calaveras-big-trees,2,CATCU_0009_20250615_1,1138,6601,True,0.447059,0.420429,0.045759,0.302752,0.497717,42.361538,23.354887,15.017296,2.236068,58.806889
2,calaveras-big-trees,3,CATCU_0009_20250615_1,1132,6603,True,0.482143,0.459324,0.055235,0.289340,0.573529,10.000000,20.300701,14.768600,3.500000,56.402573
3,calaveras-big-trees,4,CATCU_0009_20250615_1,1160,6610,True,-0.050505,0.082980,0.191337,-0.126984,0.489796,6.500000,19.293486,10.056414,1.414214,36.173195
4,calaveras-big-trees,5,CATCU_0009_20250615_1,1149,6609,True,0.478261,0.513365,0.037535,0.443299,0.602041,13.038404,17.405663,9.317881,5.000000,35.415394
